# **CPS 3320: Week 3 / HW3 — Stock Data Analysis, Econometrically Safer AI, Deep Learning, and a Time-Series Transformer**

**Instructor:** Yulia Kumar  
**Course theme:** Python libraries, data analysis, machine learning, financial time series, and deep sequence models.


**Important real-world finance note:** This notebook is for education. It does not provide investment advice. Strong-looking plots do not prove profitable trading ability. Always compare complex models against simple baselines.

## **0. Student Controls — Change These First**

Every later cell uses these values. To try your own company, change `TICKER`, `SECOND_TICKER`, and `COMPANY_NAME`, then rerun the notebook from the top.

Examples of common Yahoo Finance tickers:

| Company | Ticker |**
|---|---:|
| Apple | `AAPL` |
| Microsoft | `MSFT` |
| NVIDIA | `NVDA` |
| Tesla | `TSLA` |
| Amazon | `AMZN` |
| Alphabet / Google | `GOOGL` |

### **Student rule: Do not only run the default Microsoft example. Choose your own stock and make at least three parameter changes.**

For your reference popular_stocks = [
    "AAPL - Apple Inc.", "MSFT - Microsoft Corporation", "AMZN - Amazon.com, Inc.",
    "GOOGL - Alphabet Inc. (Class A)", "GOOG - Alphabet Inc. (Class C)", "TSLA - Tesla, Inc.",
    "FB - Meta Platforms, Inc. (formerly Facebook, Inc.)", "BRK.B - Berkshire Hathaway Inc. (Class B)",
    "NVDA - NVIDIA Corporation", "JNJ - Johnson & Johnson", "V - Visa Inc.", "WMT - Walmart Inc.",
    "JPM - JPMorgan Chase & Co.", "PG - Procter & Gamble Co.", "UNH - UnitedHealth Group Incorporated",
    "DIS - The Walt Disney Company", "HD - The Home Depot, Inc.", "MA - Mastercard Incorporated",
    "PYPL - PayPal Holdings, Inc.", "NFLX - Netflix, Inc.", "INTC - Intel Corporation",
    "CSCO - Cisco Systems, Inc.", "PEP - PepsiCo, Inc.", "KO - The Coca-Cola Company",
    "MRK - Merck & Co., Inc.", "T - AT&T Inc.", "PFE - Pfizer Inc.", "VZ - Verizon Communications Inc.",
    "XOM - Exxon Mobil Corporation", "CVX - Chevron Corporation", "ABBV - AbbVie Inc.",
    "MCD - McDonald's Corporation", "COST - Costco Wholesale Corporation", "NKE - Nike, Inc.",
    "BA - The Boeing Company", "ABT - Abbott Laboratories", "MDT - Medtronic plc",
    "ACN - Accenture plc", "CRM - Salesforce, Inc.", "ADBE - Adobe Inc.",
    "CMCSA - Comcast Corporation", "AVGO - Broadcom Inc.", "TXN - Texas Instruments Incorporated",
    "QCOM - QUALCOMM Incorporated", "LLY - Eli Lilly and Company", "ORCL - Oracle Corporation",
    "AMGN - Amgen Inc.", "HON - Honeywell International Inc.", "LMT - Lockheed Martin Corporation",
    "GE - General Electric Company"
]


In [ ]:
# ==========================================
# STUDENT CONTROL PANEL
# Change these values and rerun the notebook.
# ==========================================

STUDENT_NAME = "Yulia Kumar"      # TODO: replace with your name
TICKER = "MSFT"                   # TODO: replace with your chosen stock ticker
COMPANY_NAME = "Microsoft"        # TODO: replace with your chosen company name
SECOND_TICKER = "AAPL"            # TODO: choose a second stock for comparison
SECOND_COMPANY_NAME = "Apple"     # TODO: replace with the second company name

# Long dataset for the main company.
START_DATE = "2020-01-01"
END_DATE = "2026-05-26"

# Shorter dataset for the second file / second company.
SHORT_START_DATE = "2024-01-01"
SHORT_END_DATE = END_DATE

# Plot and output controls.
# The notebook saves visualization files as PNG images only. It does not create HTML files.
SAVE_FIGURES = True
SAVE_TO_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CPS3320_Week3_HW3_Visualizations"
LOCAL_OUTPUT_DIR = "student_graphs"
OUTPUT_DIR = LOCAL_OUTPUT_DIR  # overwritten after Drive setup if Google Drive is available

# Static Plotly image controls.
# Plotly figures are exported to PNG by Kaleido and then displayed from the saved file.
# This avoids saving HTML files.
PLOTLY_RENDERER = "png"
PLOTLY_IMAGE_FORMAT = "png"
PLOTLY_IMAGE_SCALE = 2
DISPLAY_SAVED_IMAGES = True
DISPLAY_INTERACTIVE_PLOTLY = False

# Deep-learning controls.
WINDOW_SIZE = 30                  # TODO: try 10, 20, 30, 60
TRAIN_RATIO = 0.80                # Sequential split; do not shuffle time series.
DL_TARGET = "Log_Return"          # TODO: choose "Log_Return" or "Volatility_20"
CHOSEN_RECURRENT_MODEL = "LSTM"   # Options: "RNN", "LSTM", "GRU", "BiRNN", "BiLSTM"
EPOCHS = 20                       # TODO: lower this if training is slow
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

# Optional heavy models. Keep False for quick classroom runs.
RUN_ALL_RECURRENT_MODELS = False
RUN_AUTO_ARIMA = False
RUN_PROPHET = False
RUN_DARTS_NBEATS = False
RUN_TEXT_ANALYSIS = True

# Econometric safeguards.
RUN_ADF_TESTS = True              # Augmented Dickey-Fuller tests for stationarity discussion.
REPORT_BASELINE = "PersistenceBaseline"  # Theil's U anchor for deep models.

RANDOM_SEED = 42

## **1. Install and Import Libraries**

Libraries are collections of pre-written code. In this notebook we use:

- `yfinance` to download stock data.
- `pandas` and `numpy` for data manipulation.
- `matplotlib`, `seaborn`, and `plotly` for visualization.
- `kaleido` so Plotly figures can be saved as PNG images instead of HTML files; the install cell pins a stable Plotly/Kaleido pair for classroom use.
- `statsmodels` and `scikit-learn` for statistics and machine learning.
- `tensorflow.keras` for RNN, LSTM, GRU, BiLSTM, and Transformer models.
- `nltk` and `gensim` for simple text analysis.

The optional forecasting packages `pmdarima`, `prophet`, and `darts` are not installed by default because they can take longer in Colab.


In [ ]:
# Core installation cell.
# In Google Colab this is safe to run. On a local machine, you can skip it if packages are already installed.
# Kaleido is used so Plotly can save figures as PNG images instead of HTML files.

import sys
import subprocess

INSTALL_CORE_PACKAGES = True

if INSTALL_CORE_PACKAGES:
    core_packages = [
        "yfinance",
        "pandas",
        "numpy",
        "scikit-learn",
        "tensorflow",
        "matplotlib",      # kept for a few optional package plots
        "seaborn",         # kept only for compatibility; most notebook plots now use Plotly
        "statsmodels",
        "plotly==5.24.1",
        "kaleido==0.2.1",  # stable static PNG export without browser Chrome
        "nltk",
        "gensim"
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + core_packages)
    print("Core packages installed or already available.")
else:
    print("Skipped package installation.")

In [ ]:
# Optional forecasting packages.
# Set INSTALL_OPTIONAL_FORECASTING_PACKAGES = True only if you plan to run AutoARIMA, Prophet, or Darts.

import sys
import subprocess

INSTALL_OPTIONAL_FORECASTING_PACKAGES = False

if INSTALL_OPTIONAL_FORECASTING_PACKAGES:
    optional_packages = ["pmdarima", "prophet", "u8darts[torch]"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + optional_packages)
    print("Optional forecasting packages installed.")
else:
    print("Skipped optional forecasting package installation. You can still run the main notebook.")


In [ ]:
# Imports used throughout the notebook.

import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

import matplotlib.pyplot as plt

import seaborn as sns  # compatibility import; Plotly is used for most visualizations below

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

from IPython.display import display, Image as IPyImage

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

import tensorflow as tf
from tensorflow.keras import Model, Sequential, Input
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D,
    Add,
)
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

# Mount Google Drive in Colab so saved visualization images go directly to Drive.
# Outside Colab, the notebook automatically falls back to LOCAL_OUTPUT_DIR.
if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"Google Drive mount skipped or unavailable: {exc}")

if SAVE_TO_GOOGLE_DRIVE and Path("/content/drive/MyDrive").exists():
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Visualization output folder: {OUTPUT_DIR}")

try:
    pio.renderers.default = PLOTLY_RENDERER
    print(f"Plotly renderer set to: {pio.renderers.default}")
except Exception as exc:
    print(f"Could not set Plotly renderer to {PLOTLY_RENDERER!r}: {exc}")
    print("Continuing with Plotly's default renderer.")

print("TensorFlow version:", tf.__version__)
print("Environment ready.")

## **2. Helper Functions Used Throughout the Notebook**

These functions keep the notebook clean. Students can still modify them, but most experimentation should happen in the control cells and model cells.


In [ ]:
def safe_ticker_name(ticker: str) -> str:
    """Create a file-safe ticker name."""
    return re.sub(r"[^A-Za-z0-9]+", "_", ticker).strip("_").lower()


def plot_title(base_title: str, ticker: str = None, model_name: str = None) -> str:
    """Make sure every plot title includes student name and company/ticker."""
    pieces = [base_title]
    if ticker is not None:
        pieces.append(str(ticker))
    if model_name is not None:
        pieces.append(str(model_name))
    pieces.append(f"Analyst: {STUDENT_NAME}")
    return " | ".join(pieces)


def _as_png_name(file_name: str | None, default_stem: str = "plotly_figure") -> str:
    """Force every visualization filename to be a PNG name. No HTML files are created."""
    if not file_name:
        file_name = f"{default_stem}.{PLOTLY_IMAGE_FORMAT}"
    file_name = Path(str(file_name)).name
    stem = Path(file_name).stem
    return f"{stem}.{PLOTLY_IMAGE_FORMAT}"


def save_and_show(fig_name: str | None = None):
    """Save and show a Matplotlib figure from the output folder."""
    plt.tight_layout()
    saved_path = None
    if SAVE_FIGURES and fig_name:
        png_name = _as_png_name(fig_name, default_stem="matplotlib_figure")
        saved_path = Path(OUTPUT_DIR) / png_name
        plt.savefig(saved_path, dpi=150, bbox_inches="tight")
        print(f"Saved static figure image: {saved_path}")
    plt.show()
    return saved_path


def show_plotly(fig: go.Figure, image_name: str | None = None, display_inline: bool = True):
    """Save a Plotly figure as a PNG image and display the saved image in the notebook.

    This notebook intentionally avoids Plotly HTML files. The saved PNG workflow is
    reliable for classroom use and prevents extra HTML outputs.
    """
    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        font=dict(size=12),
        margin=dict(l=40, r=30, t=80, b=40),
    )

    saved_path = None
    if SAVE_FIGURES:
        png_name = _as_png_name(image_name, default_stem="plotly_figure")
        saved_path = Path(OUTPUT_DIR) / png_name
        try:
            fig.write_image(str(saved_path), scale=PLOTLY_IMAGE_SCALE)
            print(f"Saved Plotly figure image: {saved_path}")
            if display_inline and DISPLAY_SAVED_IMAGES:
                display(IPyImage(filename=str(saved_path)))
            return saved_path
        except Exception as exc:
            print("Plotly PNG export failed. Install or update Kaleido if needed:")
            print("    !pip install -q kaleido")
            print(f"Error: {exc}")

    if display_inline and DISPLAY_INTERACTIVE_PLOTLY:
        fig.show()
    elif display_inline:
        print("Interactive Plotly display is disabled, and no saved PNG image was available to display.")
        print("Set DISPLAY_INTERACTIVE_PLOTLY=True only if your browser supports the required Plotly rendering.")
    return saved_path


def label_stock_frame(data: pd.DataFrame, ticker: str, company: str) -> pd.DataFrame:
    """Attach ticker/company labels used by combined Microsoft/Apple comparison plots."""
    out = data.copy()
    out["Ticker"] = ticker
    out["Company"] = company
    out["Ticker_Label"] = f"{company} ({ticker})"
    return out


def normalize_within_group(data: pd.DataFrame, value_col: str, group_col: str = "Ticker_Label") -> pd.Series:
    """Normalize a value column to 100 at each group's first available observation."""
    return data.groupby(group_col)[value_col].transform(lambda s: 100.0 * s / s.dropna().iloc[0])


def flatten_yfinance_columns(data: pd.DataFrame) -> pd.DataFrame:
    """Flatten yfinance MultiIndex columns if yfinance returns them."""
    data = data.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    return data


def download_stock_data(ticker: str, start: str, end: str, csv_name: str) -> pd.DataFrame:
    """Download, clean basic OHLCV stock data, and save to CSV."""
    print(f"Downloading {ticker} from {start} to {end}...")
    raw = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    raw = flatten_yfinance_columns(raw)

    if raw.empty:
        raise ValueError(f"No data returned for ticker {ticker}. Check the ticker spelling and dates.")

    data = raw.reset_index()
    data["Date"] = pd.to_datetime(data["Date"])
    data = data.sort_values("Date").reset_index(drop=True)

    expected_cols = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    keep_cols = [col for col in expected_cols if col in data.columns]
    data = data[keep_cols]

    numeric_cols = [col for col in data.columns if col != "Date"]
    data[numeric_cols] = data[numeric_cols].apply(pd.to_numeric, errors="coerce")
    data = data.ffill().dropna().reset_index(drop=True)

    data.to_csv(csv_name, index=False)
    print(f"Saved {ticker} data to {csv_name}. Rows: {len(data)}")
    return data


def print_dataset_summary(data: pd.DataFrame, label: str):
    """Quick dataset summary for students."""
    print(f"\n--- {label} ---")
    print("Shape:", data.shape)
    print("Date range:", data["Date"].min(), "to", data["Date"].max())
    print(data.head())


def plotly_corr_heatmap(corr_matrix: pd.DataFrame, title: str, image_name: str):
    """Plot a correlation matrix with Plotly and save/display it as a PNG image."""
    text = np.round(corr_matrix.values, 2)
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.index,
        zmin=-1,
        zmax=1,
        colorscale="RdBu",
        reversescale=True,
        text=text,
        texttemplate="%{text:.2f}",
        colorbar=dict(title="Correlation"),
    ))
    fig.update_layout(title=title, width=1000, height=800)
    show_plotly(fig, image_name=image_name)

## **3. Data Collection: Download One Long File and One Short/Second File**

Original task preserved: create one file with a lot of data and another file with less data or a second stock. This version does both:

- Main stock: long historical file.
- Second stock: shorter comparison file.

**Student task:** Change the tickers and dates in the control panel, rerun this cell, and inspect both CSV files.


In [ ]:
main_csv = f"{safe_ticker_name(TICKER)}_long_stock.csv"
second_csv = f"{safe_ticker_name(SECOND_TICKER)}_short_stock.csv"

main_data = download_stock_data(TICKER, START_DATE, END_DATE, main_csv)
second_data = download_stock_data(SECOND_TICKER, SHORT_START_DATE, SHORT_END_DATE, second_csv)

print_dataset_summary(main_data, f"Main long dataset: {TICKER}")
print_dataset_summary(second_data, f"Second short dataset: {SECOND_TICKER}")


## **4. Load CSV Files into DataFrames**

Original task preserved: load the CSV file into a pandas DataFrame, display the first rows, then load the second file and display its first rows too.


In [ ]:
# Load both CSV files from disk.
df = pd.read_csv(main_csv, parse_dates=["Date"])
df_second = pd.read_csv(second_csv, parse_dates=["Date"])

print(f"Main DataFrame for {TICKER}:")
print(df.head())

print(f"\nSecond DataFrame for {SECOND_TICKER}:")
print(df_second.head())


## **5. Mess Up the Data on Purpose**

Original task preserved: intentionally remove some values from the second and third records in the `Open` and `Close` columns. Then students add one more missing row and observe the result.

We do this on a copy named `df_messy` so the original downloaded data stays safe.


In [ ]:
import numpy as np

# Work on a copy so the original df remains clean.
df_messy = df.copy()

# Original task: remove values in second and third records, Open and Close columns.
rows_to_blank = [1, 2]
columns_to_blank = ["Open", "Close"]

for row in rows_to_blank:
    for col in columns_to_blank:
        df_messy.loc[row, col] = np.nan

# STUDENT TASK: remove one more record and rerun.
EXTRA_MISSING_ROW = 5  # TODO: try another row number
for col in columns_to_blank:
    df_messy.loc[EXTRA_MISSING_ROW, col] = np.nan

print("DataFrame after intentionally creating missing values:")
print(df_messy.head(10))


## **6. Detect and Clean Missing Data Without Look-Ahead Bias**

Original tasks preserved, but corrected for time-series validity:

1. Check for missing values.
2. Identify which rows and columns became missing.
3. Use forward fill as a simple temporally valid cleaning method.
4. Compare against a **past-only expanding mean** as an educational alternative.
5. Demonstrate why **global mean imputation is a leakage anti-pattern** for stock prices.

**Why this matters:** `df.mean()` uses all dates, including future dates. For a non-stationary price series, that can insert a future average price into the past and create an artificial structural break. We compute it only as a warning example and do not use it later.

In [ ]:
print("Missing values per column before cleaning:")
print(df_messy.isnull().sum())

missing_rows = df_messy[df_messy.isnull().any(axis=1)]
print("\nRows with missing values:")
print(missing_rows)

# Forward-fill method: uses the most recent available past value.
df_ffill = df_messy.copy().ffill()

print("\nMissing values after forward fill:")
print(df_ffill.isnull().sum())
print("\nForward-filled rows:")
print(df_ffill.loc[missing_rows.index, ["Date", "Open", "Close"]])

In [ ]:
# Past-only expanding mean method.
# This is more statistically defensible than a global mean because row t can only use rows before t.
df_past_mean = df_messy.copy()
for col in ["Open", "Close"]:
    past_only_mean = df_messy[col].expanding(min_periods=1).mean().shift(1)
    df_past_mean[col] = df_past_mean[col].fillna(past_only_mean)
    # If early missing values remain, use forward fill. Do not back-fill from the future.
    df_past_mean[col] = df_past_mean[col].ffill()

# Leakage demonstration only: global mean uses future information.
df_global_mean_bad = df_messy.copy()
for col in ["Open", "Close"]:
    df_global_mean_bad[col] = df_global_mean_bad[col].fillna(df_global_mean_bad[col].mean())

comparison = pd.DataFrame({
    "Date": df_messy.loc[missing_rows.index, "Date"],
    "Original_Open_With_NaN": df_messy.loc[missing_rows.index, "Open"],
    "ForwardFill_Open": df_ffill.loc[missing_rows.index, "Open"],
    "PastOnlyMean_Open": df_past_mean.loc[missing_rows.index, "Open"],
    "LeakyGlobalMean_Open_DO_NOT_USE": df_global_mean_bad.loc[missing_rows.index, "Open"],
    "Original_Close_With_NaN": df_messy.loc[missing_rows.index, "Close"],
    "ForwardFill_Close": df_ffill.loc[missing_rows.index, "Close"],
    "PastOnlyMean_Close": df_past_mean.loc[missing_rows.index, "Close"],
    "LeakyGlobalMean_Close_DO_NOT_USE": df_global_mean_bad.loc[missing_rows.index, "Close"],
})

print("Comparison of cleaning methods:")
print(comparison)
print("\nValid methods for this lab: ForwardFill and PastOnlyMean. The global mean column is shown only as an anti-pattern.")

# Keep the clean original data for the rest of the notebook.
# This avoids letting artificial missing-data experiments contaminate later analyses.
df = pd.read_csv(main_csv, parse_dates=["Date"])
df_second = pd.read_csv(second_csv, parse_dates=["Date"])

## **7. Basic Visualization with Plotly: Microsoft and Apple on the Same Plots**

The default comparison uses **Microsoft (MSFT)** and **Apple (AAPL)**. Students can change `TICKER` and `SECOND_TICKER` in the control panel to compare any two stocks. The first plot shows raw closing prices; the second plot normalizes both stocks to 100 at their first observation so that the growth paths are comparable.

In [ ]:
# Combined raw closing-price comparison for the two selected stocks.
combined_prices = pd.concat([
    label_stock_frame(df, TICKER, COMPANY_NAME),
    label_stock_frame(df_second, SECOND_TICKER, SECOND_COMPANY_NAME),
], ignore_index=True)

fig = px.line(
    combined_prices,
    x="Date",
    y="Close",
    color="Ticker_Label",
    title=plot_title("Close Price Comparison", f"{TICKER} vs {SECOND_TICKER}", "Plotly"),
    labels={"Close": "Close Price", "Ticker_Label": "Stock"},
)
fig.update_layout(width=1100, height=600)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(SECOND_TICKER)}_close_comparison.png")

In [ ]:
# Normalized comparison: both stocks start at 100.
# This is usually more informative than raw price because MSFT and AAPL can have different price levels.
combined_prices["Normalized_Close"] = normalize_within_group(combined_prices, "Close")
combined_prices["Normalized_Open"] = normalize_within_group(combined_prices, "Open")

fig = px.line(
    combined_prices,
    x="Date",
    y="Normalized_Close",
    color="Ticker_Label",
    title=plot_title("Normalized Close Comparison: Start = 100", f"{TICKER} vs {SECOND_TICKER}", "Plotly"),
    labels={"Normalized_Close": "Normalized Close", "Ticker_Label": "Stock"},
)
fig.update_layout(width=1100, height=600)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(SECOND_TICKER)}_normalized_close_comparison.png")

## **8. Moving Averages**

Original tasks preserved:

- Compute and plot 20-day and 50-day moving averages.
- Compute and plot a 100-day moving average.
- Repeat moving averages for the second dataset using `Open` price.

A moving average smooths short-term fluctuations so we can see broader trends.


In [ ]:
# Moving averages for both stocks on the same Plotly figure.
# Student task: change the windows or compare different tickers from the control panel.

def add_close_moving_averages(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    out["20_MA"] = out["Close"].rolling(window=20).mean()
    out["50_MA"] = out["Close"].rolling(window=50).mean()
    out["100_MA"] = out["Close"].rolling(window=100).mean()
    return out

# Preserve the original columns in df and df_second for later tasks.
df = add_close_moving_averages(df)
df_second = add_close_moving_averages(df_second)

combined_ma = pd.concat([
    label_stock_frame(df, TICKER, COMPANY_NAME),
    label_stock_frame(df_second, SECOND_TICKER, SECOND_COMPANY_NAME),
], ignore_index=True)

fig = go.Figure()
for label, sub in combined_ma.groupby("Ticker_Label"):
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["Close"], mode="lines", name=f"{label} Close"))
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["20_MA"], mode="lines", name=f"{label} 20-day MA", line=dict(dash="dot")))
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["50_MA"], mode="lines", name=f"{label} 50-day MA", line=dict(dash="dash")))

fig.update_layout(
    title=plot_title("Close Price and Moving Averages", f"{TICKER} vs {SECOND_TICKER}", "Plotly"),
    xaxis_title="Date",
    yaxis_title="Price",
    width=1200,
    height=700,
)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(SECOND_TICKER)}_moving_averages.png")

In [ ]:
# Normalized moving-average view for both stocks.
# This view is cleaner for comparison because both stocks start at 100.
combined_ma["Normalized_Close"] = normalize_within_group(combined_ma, "Close")
combined_ma["Normalized_20_MA"] = normalize_within_group(combined_ma, "20_MA")
combined_ma["Normalized_50_MA"] = normalize_within_group(combined_ma, "50_MA")

fig = go.Figure()
for label, sub in combined_ma.groupby("Ticker_Label"):
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["Normalized_Close"], mode="lines", name=f"{label} normalized close"))
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["Normalized_20_MA"], mode="lines", name=f"{label} normalized 20-day MA", line=dict(dash="dot")))

fig.update_layout(
    title=plot_title("Normalized Moving-Average Comparison", f"{TICKER} vs {SECOND_TICKER}", "Plotly"),
    xaxis_title="Date",
    yaxis_title="Normalized value",
    width=1200,
    height=650,
)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(SECOND_TICKER)}_normalized_moving_averages.png")

### **Student Interpretation Task: Trend Summary**

Original task preserved: summarize what is happening with the stock price over the period of analysis.

Use the next cell to generate basic evidence, then write your own 3–5 sentence interpretation below it.


In [ ]:
start_price = df["Close"].iloc[0]
end_price = df["Close"].iloc[-1]
pct_change = (end_price / start_price - 1) * 100
max_row = df.loc[df["Close"].idxmax()]
min_row = df.loc[df["Close"].idxmin()]

print(f"{TICKER} starting close: {start_price:.2f}")
print(f"{TICKER} ending close:   {end_price:.2f}")
print(f"Percent change over the selected period: {pct_change:.2f}%")
print(f"Highest close: {max_row['Close']:.2f} on {max_row['Date'].date()}")
print(f"Lowest close:  {min_row['Close']:.2f} on {min_row['Date'].date()}")

if pct_change > 0:
    print("Evidence suggests an upward overall movement across the selected period.")
elif pct_change < 0:
    print("Evidence suggests a downward overall movement across the selected period.")
else:
    print("Evidence suggests little net movement across the selected period.")


**Write your interpretation here after running the evidence cell:**

- What is the overall direction of the stock?
- Do the moving averages show a smooth trend or frequent changes?
- Are there periods where the stock behaved unusually?


## **9. Volume Analysis**

Original tasks preserved:

- Plot trading volume.
- Highlight high-volume days.
- Repeat in a similar manner for the second dataset.

High volume can indicate major events, earnings reactions, market stress, or investor attention.


In [ ]:
# Volume comparison for both stocks.
# We normalize volume by each stock's median volume so students can compare spikes across stocks.
volume_frames = []
for data, ticker, company in [(df, TICKER, COMPANY_NAME), (df_second, SECOND_TICKER, SECOND_COMPANY_NAME)]:
    temp = label_stock_frame(data, ticker, company)
    temp["Normalized_Volume"] = temp["Volume"] / temp["Volume"].median()
    temp["High_Volume"] = temp["Volume"] > temp["Volume"].quantile(0.95)
    volume_frames.append(temp)

combined_volume = pd.concat(volume_frames, ignore_index=True)

fig = go.Figure()
for label, sub in combined_volume.groupby("Ticker_Label"):
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["Normalized_Volume"], mode="lines", name=f"{label} normalized volume"))
    spikes = sub[sub["High_Volume"]]
    fig.add_trace(go.Scatter(
        x=spikes["Date"],
        y=spikes["Normalized_Volume"],
        mode="markers",
        name=f"{label} top 5% volume days",
        marker=dict(size=8, symbol="diamond"),
    ))

fig.update_layout(
    title=plot_title("Normalized Trading Volume with Spikes", f"{TICKER} vs {SECOND_TICKER}", "Plotly"),
    xaxis_title="Date",
    yaxis_title="Volume divided by median volume",
    width=1200,
    height=650,
)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(SECOND_TICKER)}_volume_spikes.png")

In [ ]:
# Table of recent high-volume days for both stocks.
high_volume_table = combined_volume[combined_volume["High_Volume"]][
    ["Ticker_Label", "Date", "Volume", "Normalized_Volume", "Close"]
].sort_values(["Ticker_Label", "Date"])

print("Recent high-volume days for the selected stocks:")
print(high_volume_table.groupby("Ticker_Label").tail(10).reset_index(drop=True))

## **10. Correlation Analysis with Plotly**

The original heatmap task is preserved, but the visualization now uses Plotly instead of Seaborn. Remember: a correlation matrix over raw prices is descriptive; it is **not** causal evidence and should not be used as a forecasting proof.

In [ ]:
# Correlation matrix using original numeric columns.
numeric_df = df.select_dtypes(include=["float64", "int64", "int32", "float32"])
corr_matrix = numeric_df.corr()

plotly_corr_heatmap(
    corr_matrix,
    title=plot_title("Correlation Matrix", TICKER, "Plotly"),
    image_name=f"{safe_ticker_name(TICKER)}_correlation_matrix_plotly.png",
)

In [ ]:
# Add additional features and recalculate correlations.
df_features = df.copy()
df_features["Daily_Return"] = df_features["Close"].pct_change()
df_features["Price_Range"] = df_features["High"] - df_features["Low"]
df_features["Price_Range_Pct"] = df_features["Price_Range"] / df_features["Close"]
df_features["Volume_Change"] = df_features["Volume"].pct_change()
df_features = df_features.replace([np.inf, -np.inf], np.nan).dropna()

updated_corr = df_features.select_dtypes(include=["float64", "int64", "int32", "float32"]).corr()

plotly_corr_heatmap(
    updated_corr,
    title=plot_title("Updated Correlation Matrix with Engineered Features", TICKER, "Plotly"),
    image_name=f"{safe_ticker_name(TICKER)}_updated_correlation_matrix_plotly.png",
)

## **11. Trend Analysis with Rolling Means**

Original task preserved: implement a method to identify and plot trends using a 50-day rolling average.


In [ ]:
# Trend analysis for both stocks on the same figure.
df["Trend_50"] = df["Close"].rolling(window=50).mean()
df_second["Trend_50"] = df_second["Close"].rolling(window=50).mean()

combined_trend = pd.concat([
    label_stock_frame(df, TICKER, COMPANY_NAME),
    label_stock_frame(df_second, SECOND_TICKER, SECOND_COMPANY_NAME),
], ignore_index=True)

fig = go.Figure()
for label, sub in combined_trend.groupby("Ticker_Label"):
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["Close"], mode="lines", name=f"{label} Close"))
    fig.add_trace(go.Scatter(x=sub["Date"], y=sub["Trend_50"], mode="lines", name=f"{label} 50-day trend", line=dict(dash="dash")))

fig.update_layout(
    title=plot_title("Trend Analysis: Close and 50-Day Moving Average", f"{TICKER} vs {SECOND_TICKER}", "Plotly"),
    xaxis_title="Date",
    yaxis_title="Price",
    width=1200,
    height=650,
)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(SECOND_TICKER)}_trend_50ma.png")

## **12. Statistical Analysis with `statsmodels`: Avoid Spurious Raw-Price Regression**

The earlier educational task asked students to fit OLS. We still do OLS, but we do it safely:

- We do **not** regress raw `Close` price on raw `Open`, `High`, `Low`, or same-day variables.
- We first discuss stationarity using the Augmented Dickey-Fuller test.
- We predict `Log_Return` using only **lagged** predictors.
- We fit on the training period only and evaluate on the later test period.

This prevents the classic financial-engineering mistake of obtaining a high \(R^2\) from shared time trends rather than genuine predictive structure.

In [ ]:
def make_causal_return_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create lagged, time-respecting predictors for return forecasting.

    Row t is interpreted as: use information available before date t to predict
    the log return realized on date t.
    """
    out = data.copy().sort_values("Date").reset_index(drop=True)

    out["Log_Return"] = np.log(out["Close"] / out["Close"].shift(1))
    out["Volume_Change"] = out["Volume"].pct_change().clip(-5, 5)
    out["Price_Range_Pct"] = (out["High"] - out["Low"]) / out["Close"]

    # Lagged predictors: these use only information from dates before the target date.
    out["Lag1_Log_Return"] = out["Log_Return"].shift(1)
    out["Lag2_Log_Return"] = out["Log_Return"].shift(2)
    out["Lag5_Mean_Return"] = out["Log_Return"].shift(1).rolling(window=5).mean()
    out["Lag20_Volatility"] = out["Log_Return"].shift(1).rolling(window=20).std() * np.sqrt(252)
    out["Lag1_Volume_Change"] = out["Volume_Change"].shift(1)
    out["Lag1_Range_Pct"] = out["Price_Range_Pct"].shift(1)

    # Previous close is used only to reconstruct one-step price predictions from predicted returns.
    out["Previous_Close"] = out["Close"].shift(1)

    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


def run_adf(series: pd.Series, label: str):
    """Run an Augmented Dickey-Fuller test and print a student-friendly summary."""
    clean = series.dropna().astype(float)
    statistic, p_value, used_lag, n_obs, critical_values, _ = adfuller(clean, autolag="AIC")
    print(f"\nADF test for {label}")
    print(f"  statistic = {statistic:.4f}")
    print(f"  p-value   = {p_value:.6f}")
    print(f"  used lags = {used_lag}, observations = {n_obs}")
    print("  Critical values:", critical_values)
    if p_value < 0.05:
        print("  Interpretation: evidence against a unit root; this series is more compatible with stationarity.")
    else:
        print("  Interpretation: cannot reject a unit root; raw regression on this series may be spurious.")
    return {"label": label, "adf_statistic": statistic, "p_value": p_value}


econometric_df = make_causal_return_features(df)

if RUN_ADF_TESTS:
    adf_close = run_adf(df["Close"], "raw Close price")
    adf_return = run_adf(econometric_df["Log_Return"], "daily Log_Return")
else:
    print("ADF tests skipped. Set RUN_ADF_TESTS = True to run them.")

ols_features = [
    "Lag1_Log_Return",
    "Lag2_Log_Return",
    "Lag5_Mean_Return",
    "Lag20_Volatility",
    "Lag1_Volume_Change",
    "Lag1_Range_Pct",
]

ols_split_idx = int(TRAIN_RATIO * len(econometric_df))
ols_train_df = econometric_df.iloc[:ols_split_idx].copy()
ols_test_df = econometric_df.iloc[ols_split_idx:].copy()

X_ols_train = sm.add_constant(ols_train_df[ols_features], has_constant="add")
y_ols_train = ols_train_df["Log_Return"]
X_ols_test = sm.add_constant(ols_test_df[ols_features], has_constant="add")
y_ols_test = ols_test_df["Log_Return"]

# HAC covariance helps the OLS summary acknowledge autocorrelation/heteroskedasticity in financial returns.
ols_model = sm.OLS(y_ols_train, X_ols_train).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(ols_model.summary())

ols_predictions = ols_model.predict(X_ols_test)
ols_zero_baseline = np.zeros_like(y_ols_test.values)
ols_persistence_baseline = ols_test_df["Lag1_Log_Return"].values

ols_rmse = math.sqrt(mean_squared_error(y_ols_test, ols_predictions))
ols_mae = mean_absolute_error(y_ols_test, ols_predictions)
ols_zero_rmse = math.sqrt(mean_squared_error(y_ols_test, ols_zero_baseline))
ols_persist_rmse = math.sqrt(mean_squared_error(y_ols_test, ols_persistence_baseline))

print("\nOut-of-sample OLS return-forecast metrics:")
print(f"  OLS MAE:                 {ols_mae:.8f}")
print(f"  OLS RMSE:                {ols_rmse:.8f}")
print(f"  Zero-return RMSE:        {ols_zero_rmse:.8f}")
print(f"  Persistence RMSE:        {ols_persist_rmse:.8f}")
print(f"  Theil's U vs persistence:{ols_rmse / ols_persist_rmse:.4f}")
print("\nInterpretation rule: Theil's U < 1 means the model beat the naive persistence baseline.")

## **13. Basics of Machine Learning with `scikit-learn`: Linear Regression on Returns**

Original task preserved: train a Linear Regression model and evaluate it with Mean Squared Error.

Corrected version:

- The target is `Log_Return`, not raw `Close`.
- Features are lagged and causal.
- The split is sequential, not shuffled.
- The model is compared against zero-return and persistence baselines.

A high \(R^2\) is not expected here. In finance, a modest result that beats a naive baseline is more meaningful than a beautiful but leaky raw-price plot.

In [ ]:
# Reuse the causal features created in the statsmodels section.
ml_df = econometric_df.copy()
ml_features = ols_features
ml_target = "Log_Return"

split_idx = int(TRAIN_RATIO * len(ml_df))
X_train_ml = ml_df.loc[:split_idx - 1, ml_features]
X_test_ml = ml_df.loc[split_idx:, ml_features]
y_train_ml = ml_df.loc[:split_idx - 1, ml_target]
y_test_ml = ml_df.loc[split_idx:, ml_target]

linear_model = LinearRegression()
linear_model.fit(X_train_ml, y_train_ml)

ml_predictions = linear_model.predict(X_test_ml)
ml_zero_baseline = np.zeros_like(y_test_ml.values)
ml_persistence_baseline = ml_df.loc[split_idx:, "Lag1_Log_Return"].values

ml_mse = mean_squared_error(y_test_ml, ml_predictions)
ml_mae = mean_absolute_error(y_test_ml, ml_predictions)
ml_rmse = math.sqrt(ml_mse)
ml_r2 = r2_score(y_test_ml, ml_predictions)
ml_zero_rmse = math.sqrt(mean_squared_error(y_test_ml, ml_zero_baseline))
ml_persist_rmse = math.sqrt(mean_squared_error(y_test_ml, ml_persistence_baseline))
ml_theils_u = ml_rmse / ml_persist_rmse if ml_persist_rmse > 0 else np.nan

ml_metrics = pd.DataFrame([
    {"Model": "LinearRegression", "Target": ml_target, "MAE": ml_mae, "RMSE": ml_rmse, "R2": ml_r2, "Theils_U_vs_Persistence": ml_theils_u},
    {"Model": "ZeroReturnBaseline", "Target": ml_target, "MAE": mean_absolute_error(y_test_ml, ml_zero_baseline), "RMSE": ml_zero_rmse, "R2": r2_score(y_test_ml, ml_zero_baseline), "Theils_U_vs_Persistence": ml_zero_rmse / ml_persist_rmse},
    {"Model": "PersistenceBaseline", "Target": ml_target, "MAE": mean_absolute_error(y_test_ml, ml_persistence_baseline), "RMSE": ml_persist_rmse, "R2": r2_score(y_test_ml, ml_persistence_baseline), "Theils_U_vs_Persistence": 1.0},
])

print("Linear regression and baseline metrics:")
print(ml_metrics.sort_values("RMSE").reset_index(drop=True))

In [ ]:
# Visualize one-step linear regression predictions after converting predicted returns back to prices.
ml_test_dates = ml_df.loc[split_idx:, "Date"]
actual_test_prices = ml_df.loc[split_idx:, "Close"].values
previous_test_prices = ml_df.loc[split_idx:, "Previous_Close"].values
predicted_test_prices = previous_test_prices * np.exp(ml_predictions)
persistence_prices = previous_test_prices  # random-walk price baseline: tomorrow's price equals previous close

price_reconstruction_df = pd.DataFrame({
    "Date": ml_test_dates.values,
    "Actual Close": actual_test_prices,
    "Predicted Close from Linear Return Model": predicted_test_prices,
    "Naive Random-Walk Price Baseline": persistence_prices,
})

fig = px.line(
    price_reconstruction_df,
    x="Date",
    y=["Actual Close", "Predicted Close from Linear Return Model", "Naive Random-Walk Price Baseline"],
    title=plot_title("One-Step Price Reconstruction from Return Predictions", TICKER, "scikit-learn + Plotly"),
    labels={"value": "Close Price", "variable": "Series"},
)
fig.update_layout(width=1200, height=650)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_linear_return_price_reconstruction.png")

## **14. Future Prediction Demonstration**

Original task preserved: predict future values. Corrected version:

- The model predicts future **log returns**, not raw prices directly.
- Predicted returns are compounded into a toy future price path.
- This is still only a classroom demonstration, not a trading system.

**Student task:** Change `n_future_days` and explain why uncertainty grows as the horizon gets longer.

In [ ]:
n_future_days = 50  # TODO: try 10, 30, 100
last_date = ml_df["Date"].max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=n_future_days, freq="B")

# Start from the last valid feature row. The feature-updating loop below is intentionally simple.
current_features = ml_df[ml_features].iloc[-1].copy()
current_price = float(df["Close"].iloc[-1])
future_rows = []

recent_returns = list(ml_df["Log_Return"].tail(20).values)
for i, future_date in enumerate(future_dates):
    feature_frame = pd.DataFrame([current_features], columns=ml_features)
    predicted_return = float(linear_model.predict(feature_frame)[0])
    current_price = current_price * math.exp(predicted_return)

    future_rows.append({
        "Date": future_date,
        "Predicted_Log_Return": predicted_return,
        "Predicted_Close_From_Returns": current_price,
    })

    # Toy causal update: use the predicted return as the newest lag.
    recent_returns.append(predicted_return)
    recent_returns = recent_returns[-20:]
    current_features["Lag2_Log_Return"] = current_features["Lag1_Log_Return"]
    current_features["Lag1_Log_Return"] = predicted_return
    current_features["Lag5_Mean_Return"] = np.mean(recent_returns[-5:])
    current_features["Lag20_Volatility"] = np.std(recent_returns[-20:]) * np.sqrt(252)
    # Keep non-return exogenous features unchanged for the toy demonstration.

future_df = pd.DataFrame(future_rows)
print(future_df.head())

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["Date"], y=df["Close"], mode="lines", name="Historical Close"))
fig.add_trace(go.Scatter(x=future_df["Date"], y=future_df["Predicted_Close_From_Returns"], mode="lines", name="Future Price Path from Predicted Returns"))
fig.update_layout(
    title=plot_title("Simple Future Prediction Demonstration", TICKER, "Linear Return Model + Plotly"),
    xaxis_title="Date",
    yaxis_title="Close Price",
    width=1200,
    height=650,
)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_future_return_based_prediction.png")

## **15. Two-Stock 2D Visualization: Microsoft and Apple in One Figure**

All 3D graphs have been removed. This section uses a 2D analog: normalized price paths for the two selected stocks on the same axes.

The visualization still compares the same idea as the old 3D version—relative stock movement over time—but it is easier to read, faster to export, and safer for Colab/browser environments.


In [ ]:
# 2D comparison of the two selected stocks.
# This replaces the earlier 3D normalized-price-path figure.
# No HTML file is created.

two_d_frames = []
for data, ticker, company in [(df, TICKER, COMPANY_NAME), (df_second, SECOND_TICKER, SECOND_COMPANY_NAME)]:
    temp = label_stock_frame(data, ticker, company)
    temp = temp.sort_values("Date").reset_index(drop=True)
    temp["Normalized_Close"] = 100.0 * temp["Close"] / temp["Close"].iloc[0]
    two_d_frames.append(temp)

combined_2d = pd.concat(two_d_frames, ignore_index=True)

fig2d = go.Figure()
for label, sub in combined_2d.groupby("Ticker_Label"):
    fig2d.add_trace(go.Scatter(
        x=sub["Date"],
        y=sub["Normalized_Close"],
        mode="lines",
        name=label,
        hovertemplate="Stock=%{fullData.name}<br>Date=%{x}<br>Normalized Close=%{y:.2f}<extra></extra>",
    ))

fig2d.update_layout(
    title=plot_title("2D Normalized Price Paths", f"{TICKER} vs {SECOND_TICKER}", "Start = 100"),
    xaxis_title="Date",
    yaxis_title="Normalized Close: Start = 100",
    width=1100,
    height=650,
)

show_plotly(
    fig2d,
    image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(SECOND_TICKER)}_2d_normalized_close.png",
    display_inline=True,
)

print("2D normalized price-path comparison saved and displayed. All 3D graph output was removed.")


## **16. Optional Classical Forecasting: AutoARIMA on Returns**

Original task preserved: run a predefined model, adjust numerical parameters, rerun, and compare results.

Corrected version: AutoARIMA is applied to `Log_Return` rather than raw `Close`, then compared with zero-return and persistence baselines.

Set `RUN_AUTO_ARIMA = True` in the control panel and install optional packages if you want to run this cell.

In [ ]:
if RUN_AUTO_ARIMA:
    try:
        import pmdarima as pm
    except ImportError as exc:
        raise ImportError("Install pmdarima first by setting INSTALL_OPTIONAL_FORECASTING_PACKAGES = True.") from exc

    arima_series = econometric_df["Log_Return"].astype(float)
    arima_dates = econometric_df["Date"]
    arima_train_size = int(TRAIN_RATIO * len(arima_series))
    arima_train = arima_series.iloc[:arima_train_size]
    arima_test = arima_series.iloc[arima_train_size:]

    # STUDENT TASK: change seasonal, max_p, max_q, or stepwise.
    arima_model = pm.auto_arima(
        arima_train,
        seasonal=False,
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        max_p=5,
        max_q=5
    )

    arima_pred = arima_model.predict(n_periods=len(arima_test))
    arima_rmse = math.sqrt(mean_squared_error(arima_test, arima_pred))
    arima_persistence = econometric_df["Lag1_Log_Return"].iloc[arima_train_size:].values
    arima_persist_rmse = math.sqrt(mean_squared_error(arima_test, arima_persistence))
    print(arima_model.summary())
    print(f"AutoARIMA return RMSE: {arima_rmse:.8f}")
    print(f"Persistence return RMSE: {arima_persist_rmse:.8f}")
    print(f"Theil's U vs persistence: {arima_rmse / arima_persist_rmse:.4f}")

    arima_plot_df = pd.DataFrame({
        "Date": arima_dates,
        "Train Log Return": np.r_[arima_train.values, [np.nan] * len(arima_test)],
        "Test Log Return": np.r_[[np.nan] * len(arima_train), arima_test.values],
        "AutoARIMA Prediction": np.r_[[np.nan] * len(arima_train), np.asarray(arima_pred)],
    })
    fig = px.line(
        arima_plot_df,
        x="Date",
        y=["Train Log Return", "Test Log Return", "AutoARIMA Prediction"],
        title=plot_title("AutoARIMA Forecast on Log Returns", TICKER, "pmdarima + Plotly"),
        labels={"value": "Log Return", "variable": "Series"},
    )
    fig.update_layout(width=1200, height=650)
    show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_auto_arima_log_returns.png")
else:
    print("AutoARIMA skipped. Set RUN_AUTO_ARIMA = True and install optional packages to run it.")

## **17. Optional Predefined Deep Forecasting: Darts N-BEATS on Returns**

Original task preserved: run Darts/N-BEATS, add your name, company, model name, and adjust epochs, plot size, and parameters.

Corrected version: the optional predefined model is applied to `Log_Return` rather than raw price.

Set `RUN_DARTS_NBEATS = True` in the control panel and install optional packages if you want to run this cell.

In [ ]:
if RUN_DARTS_NBEATS:
    try:
        from darts import TimeSeries
        from darts.models import NBEATSModel
    except ImportError as exc:
        raise ImportError("Install Darts first by setting INSTALL_OPTIONAL_FORECASTING_PACKAGES = True.") from exc

    darts_df = econometric_df[["Date", "Log_Return"]].copy()
    series = TimeSeries.from_dataframe(darts_df, time_col="Date", value_cols="Log_Return")
    train_series, val_series = series.split_after(TRAIN_RATIO)

    # STUDENT TASK: lower n_epochs if training is slow; try input_chunk_length values 12, 24, 30.
    nbeats_model = NBEATSModel(
        input_chunk_length=30,
        output_chunk_length=10,
        n_epochs=10,
        random_state=RANDOM_SEED,
        force_reset=True
    )
    nbeats_model.fit(train_series, verbose=True)
    nbeats_pred = nbeats_model.predict(len(val_series))

    darts_plot_df = pd.concat([
        pd.DataFrame({"Date": train_series.time_index, "Log_Return": train_series.values().ravel(), "Series": "Train Log Return"}),
        pd.DataFrame({"Date": val_series.time_index, "Log_Return": val_series.values().ravel(), "Series": "Actual Test Log Return"}),
        pd.DataFrame({"Date": nbeats_pred.time_index, "Log_Return": nbeats_pred.values().ravel(), "Series": "N-BEATS Prediction"}),
    ], ignore_index=True)

    fig = px.line(
        darts_plot_df,
        x="Date",
        y="Log_Return",
        color="Series",
        title=plot_title("Darts Forecast on Log Returns", TICKER, "N-BEATS + Plotly"),
    )
    fig.update_layout(width=1200, height=650)
    show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_darts_nbeats_log_returns.png")
else:
    print("Darts N-BEATS skipped. Set RUN_DARTS_NBEATS = True and install optional packages to run it.")

## **18. Optional Prophet Forecasting on Returns**

Original task preserved: run Prophet, add your name, model name, company, and adjust the plot.

Corrected version: Prophet is applied to `Log_Return`. A trend model on raw stock price can easily look impressive for the wrong reason.

Set `RUN_PROPHET = True` in the control panel and install optional packages if you want to run this cell.

In [ ]:
if RUN_PROPHET:
    try:
        from prophet import Prophet
        from prophet.plot import plot_plotly, plot_components_plotly
    except ImportError as exc:
        raise ImportError("Install Prophet first by setting INSTALL_OPTIONAL_FORECASTING_PACKAGES = True.") from exc

    prophet_df = econometric_df[["Date", "Log_Return"]].rename(columns={"Date": "ds", "Log_Return": "y"})
    prophet_model = Prophet()
    prophet_model.fit(prophet_df)

    future = prophet_model.make_future_dataframe(periods=50, freq="B")
    forecast = prophet_model.predict(future)

    fig_forecast = plot_plotly(prophet_model, forecast)
    fig_forecast.update_layout(title=plot_title("Prophet Forecast on Log Returns", TICKER, "Prophet + Plotly"))
    show_plotly(fig_forecast, image_name=f"{safe_ticker_name(TICKER)}_prophet_log_return_forecast.png")

    fig_components = plot_components_plotly(prophet_model, forecast)
    fig_components.update_layout(title=plot_title("Prophet Components for Log Returns", TICKER, "Prophet + Plotly"))
    show_plotly(fig_components, image_name=f"{safe_ticker_name(TICKER)}_prophet_log_return_components.png")
else:
    print("Prophet skipped. Set RUN_PROPHET = True and install optional packages to run it.")

## **19. Text Analysis: Sentiment of a Company Article**

Original task preserved: find an article or paragraph mentioning your company, save it as a string, and run a sentiment analysis.

**Student task:** Replace `ARTICLE_TEXT` with a short article excerpt, press release, or your own summary about your chosen company. Avoid pasting a very long copyrighted article.


In [ ]:
ARTICLE_TEXT = f"""
{COMPANY_NAME} is an important technology company in the stock market. Investors often watch its earnings,
cloud growth, artificial intelligence strategy, product launches, and operating expenses. Some analysts are optimistic
about future growth, while others are cautious about valuation, competition, and macroeconomic risk.
"""

if RUN_TEXT_ANALYSIS:
    import nltk
    nltk.download("vader_lexicon", quiet=True)
    from nltk.sentiment.vader import SentimentIntensityAnalyzer

    analyzer = SentimentIntensityAnalyzer()
    sentiment_scores = analyzer.polarity_scores(ARTICLE_TEXT)

    print("Article text:")
    print(ARTICLE_TEXT[:1000])
    print("\nSentiment scores:")
    print(sentiment_scores)

    if sentiment_scores["compound"] >= 0.05:
        label = "mostly positive"
    elif sentiment_scores["compound"] <= -0.05:
        label = "mostly negative"
    else:
        label = "mostly neutral"

    print(f"Interpretation: The text sentiment is {label}.")
else:
    print("Text sentiment skipped. Set RUN_TEXT_ANALYSIS = True to run it.")


## **20. Topic Modeling with Gensim or a Safe Fallback**

Original task preserved: use topic modeling to discover major themes in company-related text.

This cell first tries `gensim`. If the environment has a compatibility issue, it falls back to scikit-learn's LDA implementation.


In [ ]:
if RUN_TEXT_ANALYSIS:
    documents = [
        ARTICLE_TEXT,
        f"{COMPANY_NAME} reported business updates, product development, and market activity.",
        f"Investors evaluate {COMPANY_NAME} using revenue, profit, risk, innovation, and future growth expectations.",
    ]

    try:
        import gensim
        from gensim import corpora
        import nltk
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords

        stop_words = set(stopwords.words("english"))
        tokenized_docs = []
        for doc in documents:
            tokens = [word.lower() for word in re.findall(r"[A-Za-z]+", doc)]
            tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
            tokenized_docs.append(tokens)

        dictionary = corpora.Dictionary(tokenized_docs)
        corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

        lda_model = gensim.models.LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=2,
            random_state=RANDOM_SEED,
            passes=10
        )

        print("Gensim LDA topics:")
        for idx, topic in lda_model.print_topics(num_words=6):
            print(f"Topic {idx}: {topic}")

    except Exception as exc:
        print("Gensim was unavailable or incompatible. Using scikit-learn LDA fallback.")
        print("Reason:", exc)

        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.decomposition import LatentDirichletAllocation

        vectorizer = CountVectorizer(stop_words="english", max_features=50)
        document_term_matrix = vectorizer.fit_transform(documents)
        lda = LatentDirichletAllocation(n_components=2, random_state=RANDOM_SEED)
        lda.fit(document_term_matrix)

        words = np.array(vectorizer.get_feature_names_out())
        for topic_idx, topic in enumerate(lda.components_):
            top_words = words[topic.argsort()[-6:][::-1]]
            print(f"Topic {topic_idx}: {', '.join(top_words)}")
else:
    print("Topic modeling skipped. Set RUN_TEXT_ANALYSIS = True to run it.")


# **Part II: Deep Sequence Models for Financial Time Series**

The earlier cells predicted or analyzed raw prices. In this section we follow a more realistic financial modeling idea:

- Use **log returns** instead of raw prices.
- Compute **realized volatility**.
- Create **sliding windows** of past observations.
- Train sequence models to predict the next time step.

Models included:

1. Simple RNN
2. LSTM
3. GRU
4. Bidirectional RNN
5. Bidirectional LSTM
6. Time-Series Transformer Encoder at the end


## **21. Feature Engineering for Deep Learning**

We engineer causal stationary or semi-stationary features:

- `Log_Return`: daily log return.
- `Volatility_20`: annualized 20-day realized volatility.
- `Abs_Log_Return`: magnitude of the return, useful for volatility modeling.
- `RSI`: Relative Strength Index.
- `Price_Range_Pct`: intraday range divided by close price.
- `Volume_Change`: daily percentage change in volume.

**Important target distinction:**

- `Log_Return` is roughly zero-centered and noisy. The output layer can be linear.
- `Volatility_20` is non-negative and persistent. The output layer uses a non-negative activation.

**Student task:** Change `DL_TARGET` in the control panel to `Volatility_20`, rerun, and explain why the behavior is different. Do not compare log-return RMSE and volatility RMSE as if they were the same forecasting problem.

In [ ]:
def add_financial_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create features for sequence models.

    The rolling features are backward-looking. After this, scalers are fit only on
    the training period, so no fitted transformation uses validation/test data.
    """
    out = data.copy().sort_values("Date").reset_index(drop=True)

    out["Log_Return"] = np.log(out["Close"] / out["Close"].shift(1))
    out["Abs_Log_Return"] = out["Log_Return"].abs()
    out["Volatility_20"] = out["Log_Return"].rolling(window=20).std() * np.sqrt(252)

    delta = out["Close"].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    out["RSI"] = 100 - (100 / (1 + rs))

    out["Price_Range_Pct"] = (out["High"] - out["Low"]) / out["Close"]
    out["Volume_Change"] = out["Volume"].pct_change().clip(-5, 5)

    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


def get_target_config(target: str) -> dict:
    """Return target-specific modeling settings.

    We keep this small and transparent so students can inspect how target choice
    changes the model's output activation and loss.
    """
    configs = {
        "Log_Return": {
            "output_activation": "linear",
            "loss": tf.keras.losses.Huber(delta=0.05),
            "loss_label": "Huber loss on scaled log returns",
            "description": "zero-centered return target; linear output is allowed",
        },
        "Volatility_20": {
            "output_activation": "softplus",
            "loss": tf.keras.losses.Huber(delta=0.05),
            "loss_label": "Huber loss on scaled realized volatility",
            "description": "non-negative volatility target; softplus output enforces non-negative predictions",
        },
    }
    if target not in configs:
        raise ValueError(f"DL_TARGET must be one of {list(configs.keys())}, not {target!r}.")
    return configs[target]


dl_df = add_financial_features(df)
TARGET_CONFIG = get_target_config(DL_TARGET)

DL_FEATURES = ["Log_Return", "Abs_Log_Return", "RSI", "Volatility_20", "Price_Range_Pct", "Volume_Change"]
if DL_TARGET not in dl_df.columns:
    raise ValueError(f"DL_TARGET must be one of these columns: {list(dl_df.columns)}")

print(dl_df[["Date", "Close"] + DL_FEATURES].head())
print("\nDeep-learning target:", DL_TARGET)
print("Target configuration:", TARGET_CONFIG["description"])
print("Loss used for neural networks:", TARGET_CONFIG["loss_label"])
print("Deep-learning features:", DL_FEATURES)

## **22. Sequence Generation for Deep Learning**

Neural networks expect a 3D tensor:

\[
X = [\text{samples}, \text{time steps}, \text{features}]
\]

The sliding-window function below takes `WINDOW_SIZE` days of history and predicts the next day's target.

**Student task:** Change `WINDOW_SIZE` in the control panel and compare training speed and validation loss.


In [ ]:
def create_sequences(features_data: np.ndarray, target_data: np.ndarray, window_size: int):
    X, y = [], []
    for i in range(len(features_data) - window_size):
        X.append(features_data[i:i + window_size])
        y.append(target_data[i + window_size])
    return np.array(X), np.array(y)


def evaluate_array_predictions(actual: np.ndarray, pred: np.ndarray, model_name: str, baseline_rmse: float | None = None):
    """Evaluate any prediction vector and store a comparable result row."""
    actual = np.asarray(actual).ravel()
    pred = np.asarray(pred).ravel()
    mae = mean_absolute_error(actual, pred)
    rmse = math.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    theils_u = rmse / baseline_rmse if baseline_rmse is not None and baseline_rmse > 0 else np.nan

    result = {
        "Model": model_name,
        "Target": DL_TARGET,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Theils_U_vs_Persistence": theils_u,
        "Beats_Persistence": bool(theils_u < 1) if np.isfinite(theils_u) else None,
    }
    model_results.append(result)
    predictions_by_model[model_name] = {"actual": actual, "pred": pred}
    return result

# Sequential train/test split at the raw-row level.
train_cutoff = int(TRAIN_RATIO * len(dl_df))
if train_cutoff <= WINDOW_SIZE:
    raise ValueError("Training cutoff is too small for the selected WINDOW_SIZE. Use more data or a smaller WINDOW_SIZE.")

# Fit scalers only on the training period to avoid test leakage.
scaler_features = MinMaxScaler(feature_range=(0, 1))
scaler_target = MinMaxScaler(feature_range=(0, 1))

scaler_features.fit(dl_df.loc[:train_cutoff - 1, DL_FEATURES])
scaler_target.fit(dl_df.loc[:train_cutoff - 1, [DL_TARGET]])

scaled_features = scaler_features.transform(dl_df[DL_FEATURES])
scaled_target = scaler_target.transform(dl_df[[DL_TARGET]])

X, y = create_sequences(scaled_features, scaled_target, WINDOW_SIZE)

# First test target corresponds to raw row train_cutoff.
sequence_split = train_cutoff - WINDOW_SIZE
X_train, X_test = X[:sequence_split], X[sequence_split:]
y_train, y_test = y[:sequence_split], y[sequence_split:]

test_dates = dl_df["Date"].iloc[train_cutoff:train_cutoff + len(y_test)].reset_index(drop=True)
actual_test_target = dl_df[DL_TARGET].iloc[train_cutoff:train_cutoff + len(y_test)].to_numpy()

print(f"Training shape: X={X_train.shape}, y={y_train.shape}")
print(f"Testing shape:  X={X_test.shape}, y={y_test.shape}")
print(f"First test date: {test_dates.iloc[0].date()} | Last test date: {test_dates.iloc[-1].date()}")

# Baselines: every deep model must beat these to be meaningful.
model_results = []
predictions_by_model = {}

persistence_pred = dl_df[DL_TARGET].iloc[train_cutoff - 1:train_cutoff - 1 + len(y_test)].to_numpy()
persistence_result = evaluate_array_predictions(actual_test_target, persistence_pred, "PersistenceBaseline")
baseline_rmse_for_theils_u = persistence_result["RMSE"]
model_results[-1]["Theils_U_vs_Persistence"] = 1.0
model_results[-1]["Beats_Persistence"] = False

if DL_TARGET == "Log_Return":
    zero_return_pred = np.zeros_like(actual_test_target)
    evaluate_array_predictions(actual_test_target, zero_return_pred, "ZeroReturnBaseline", baseline_rmse_for_theils_u)

print("\nBaseline metrics before training any neural network:")
print(pd.DataFrame(model_results).sort_values("RMSE").reset_index(drop=True))
print("\nInterpretation rule: Theil's U < 1 means the model beat the persistence baseline.")

## **23. Recurrent Model Factory: RNN, LSTM, GRU, BiRNN, BiLSTM**

Original deep-learning task preserved: students can choose a model architecture and hyperparameters.

**Important warning about SimpleRNN:** A vanilla `SimpleRNN` often loses long-range information because gradients can decay exponentially across time. With a 30-day or 60-day window, it is mainly a teaching baseline. For serious sequence modeling, compare it with `LSTM` or `GRU`.

**Student task:** Change `CHOSEN_RECURRENT_MODEL`, `EPOCHS`, `BATCH_SIZE`, or `LEARNING_RATE` in the control panel.

In [ ]:
def build_recurrent_model(architecture: str = "LSTM", input_shape=None) -> tf.keras.Model:
    if input_shape is None:
        input_shape = (WINDOW_SIZE, len(DL_FEATURES))

    if architecture == "RNN" and input_shape[0] > 15:
        print(
            "WARNING: SimpleRNN with WINDOW_SIZE > 15 is likely to suffer from vanishing gradients. "
            "Keep it as a baseline, but compare it with LSTM or GRU."
        )

    if architecture in {"BiRNN", "BiLSTM"}:
        print(
            "NOTE: Bidirectional models read the past window in both directions. This is not future leakage "
            "because the whole input window is historical, but it is less like a strict real-time causal filter."
        )

    model = Sequential(name=f"{architecture}_forecast_model")
    model.add(Input(shape=input_shape))

    if architecture == "RNN":
        model.add(SimpleRNN(64, return_sequences=False))
    elif architecture == "LSTM":
        model.add(LSTM(64, return_sequences=False))
    elif architecture == "GRU":
        model.add(GRU(64, return_sequences=False))
    elif architecture == "BiRNN":
        model.add(Bidirectional(SimpleRNN(64, return_sequences=False)))
    elif architecture == "BiLSTM":
        model.add(Bidirectional(LSTM(64, return_sequences=False)))
    else:
        raise ValueError("Invalid architecture. Choose: RNN, LSTM, GRU, BiRNN, or BiLSTM.")

    model.add(Dropout(0.20))
    model.add(Dense(1, activation=TARGET_CONFIG["output_activation"]))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=TARGET_CONFIG["loss"],
        metrics=["mae"]
    )
    return model

# Test the builder.
temp_model = build_recurrent_model("LSTM", input_shape=(X_train.shape[1], X_train.shape[2]))
temp_model.summary()

## **24. Train the Selected Recurrent Model**

Watch `val_loss`. If training loss decreases while validation loss increases, the model may be overfitting.


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

print(f"Training recurrent model: {CHOSEN_RECURRENT_MODEL}")
recurrent_model = build_recurrent_model(
    CHOSEN_RECURRENT_MODEL,
    input_shape=(X_train.shape[1], X_train.shape[2])
)

recurrent_history = recurrent_model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1,
    shuffle=False,
    callbacks=callbacks
)

In [ ]:
def plot_training_history(history, model_name: str):
    """Plot training and validation loss without using Plotly Express labels.

    Plotly Express can fail if a non-string object, such as a Keras loss object,
    accidentally enters its label dictionary. This graph_objects version is more
    explicit and robust for classroom notebooks.
    """
    epochs = np.arange(1, len(history.history["loss"]) + 1)
    loss_label = str(TARGET_CONFIG.get("loss_label", "Loss"))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=epochs,
        y=history.history["loss"],
        mode="lines+markers",
        name="Training Loss",
        hovertemplate="Epoch=%{x}<br>Training Loss=%{y:.6f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=epochs,
        y=history.history["val_loss"],
        mode="lines+markers",
        name="Validation Loss",
        hovertemplate="Epoch=%{x}<br>Validation Loss=%{y:.6f}<extra></extra>",
    ))

    fig.update_layout(
        title=plot_title("Loss Curve", TICKER, f"{model_name} + Plotly"),
        xaxis_title="Epoch",
        yaxis_title=loss_label,
        width=1000,
        height=500,
    )
    show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(model_name)}_loss_curve.png")

plot_training_history(recurrent_history, CHOSEN_RECURRENT_MODEL)


## **25. Evaluate Deep Model Predictions Against Baselines**

The model predicts scaled targets, so we inverse-transform predictions back to real log returns or volatility values.

We also report **Theil's U versus persistence**:

\[
U =
rac{	ext{RMSE}_{	ext{model}}}{	ext{RMSE}_{	ext{persistence baseline}}}
\]

A value below 1 means the model beat the naive last-value baseline. A value above 1 means the complex model did worse than a simple copy-paste rule.

In [ ]:
def evaluate_sequence_model(model: tf.keras.Model, model_name: str):
    pred_scaled = model.predict(X_test, verbose=0)
    pred = scaler_target.inverse_transform(pred_scaled).ravel()
    actual = scaler_target.inverse_transform(y_test).ravel()

    result = evaluate_array_predictions(actual, pred, model_name, baseline_rmse_for_theils_u)

    print(pd.DataFrame([result]))
    if np.isfinite(result["Theils_U_vs_Persistence"]):
        if result["Theils_U_vs_Persistence"] < 1:
            print(f"{model_name} beat the persistence baseline.")
        else:
            print(f"{model_name} did NOT beat the persistence baseline.")
    return actual, pred, result

actual_target, recurrent_pred, recurrent_result = evaluate_sequence_model(recurrent_model, CHOSEN_RECURRENT_MODEL)

target_prediction_df = pd.DataFrame({
    "Date": test_dates.values,
    f"Actual {DL_TARGET}": actual_target,
    f"Predicted {DL_TARGET}": recurrent_pred,
})

fig = px.line(
    target_prediction_df,
    x="Date",
    y=[f"Actual {DL_TARGET}", f"Predicted {DL_TARGET}"],
    title=plot_title(f"Actual vs Predicted {DL_TARGET}", TICKER, f"{CHOSEN_RECURRENT_MODEL} + Plotly"),
    labels={"value": DL_TARGET, "variable": "Series"},
)
fig.update_layout(width=1200, height=600)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(CHOSEN_RECURRENT_MODEL)}_target_prediction.png")

## **26. Convert Predicted Log Returns Back to a Price Path**

This visualization is only meaningful when `DL_TARGET = "Log_Return"`. If you change the target to volatility, this cell will skip price reconstruction.


In [ ]:
def plot_reconstructed_price_path(predicted_log_returns, model_name: str):
    if DL_TARGET != "Log_Return":
        print("Price-path reconstruction skipped because DL_TARGET is not Log_Return.")
        return

    anchor_price = dl_df["Close"].iloc[train_cutoff - 1]
    actual_log_returns = predictions_by_model[model_name]["actual"]
    persistence_log_returns = predictions_by_model["PersistenceBaseline"]["pred"]

    actual_prices = anchor_price * np.exp(np.cumsum(actual_log_returns))
    predicted_prices = anchor_price * np.exp(np.cumsum(predicted_log_returns))
    persistence_prices = anchor_price * np.exp(np.cumsum(persistence_log_returns))
    random_walk_prices = np.repeat(anchor_price, len(actual_prices))

    reconstructed_df = pd.DataFrame({
        "Date": test_dates.values,
        "Actual Reconstructed Price": actual_prices,
        f"Predicted Reconstructed Price ({model_name})": predicted_prices,
        "Persistence Baseline Price Path": persistence_prices,
        "Zero-Return Random Walk Baseline": random_walk_prices,
    })

    fig = px.line(
        reconstructed_df,
        x="Date",
        y=[
            "Actual Reconstructed Price",
            f"Predicted Reconstructed Price ({model_name})",
            "Persistence Baseline Price Path",
            "Zero-Return Random Walk Baseline",
        ],
        title=plot_title("Out-of-Sample Price Path from Predicted Log Returns", TICKER, f"{model_name} + Plotly"),
        labels={"value": "Price", "variable": "Series"},
    )
    fig.update_layout(width=1200, height=650)
    show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_{safe_ticker_name(model_name)}_reconstructed_price_path.png")

plot_reconstructed_price_path(recurrent_pred, CHOSEN_RECURRENT_MODEL)

## **27. Optional: Compare All Five Recurrent Architectures**

Set `RUN_ALL_RECURRENT_MODELS = True` in the control panel to train RNN, LSTM, GRU, BiRNN, and BiLSTM in one loop.

This preserves the original objective of comparing five neural sequence models.


In [ ]:
if RUN_ALL_RECURRENT_MODELS:
    recurrent_architectures = ["RNN", "LSTM", "GRU", "BiRNN", "BiLSTM"]

    for arch in recurrent_architectures:
        print("\n" + "=" * 70)
        print(f"Training {arch}")
        tf.keras.backend.clear_session()
        candidate = build_recurrent_model(arch, input_shape=(X_train.shape[1], X_train.shape[2]))
        hist = candidate.fit(
            X_train,
            y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            verbose=0,
            shuffle=False,
            callbacks=callbacks,
        )
        evaluate_sequence_model(candidate, arch)

    results_df = pd.DataFrame(model_results).drop_duplicates(subset=["Model", "Target"], keep="last")
    print(results_df.sort_values("RMSE"))
else:
    print("All-model recurrent comparison skipped. Set RUN_ALL_RECURRENT_MODELS = True to run it.")


# **Part III: Time-Series Transformer Encoder**

## **28. Why a Transformer Here?**

A standard NLP Transformer is not the right mental model for this task. In NLP, an `Embedding` layer maps discrete token IDs to vectors. Here the inputs are already continuous numerical time-series features.

The suitable classroom version is a **lightweight time-series Transformer encoder**:

- It receives a numeric sequence of shape `[window_size, number_of_features]`.
- A `Dense` layer projects continuous numerical features into `model_dim`.
- Fixed sinusoidal positional encoding tells the model the order of days.
- Multi-head attention learns which historical days matter most.
- A feed-forward block transforms the attended features.
- A target-specific regression head predicts the next log return or volatility value.

**Student task:** Try changing the number of heads, Transformer blocks, model dimension, dropout, and target variable. Then compare Theil's U against the naive baseline.

In [ ]:
# Transformer hyperparameters.
# These are intentionally small for classroom use.
TRANSFORMER_MODEL_DIM = 32      # TODO: try 16, 32, 64
TRANSFORMER_HEADS = 4           # TODO: try 2, 4, 8
TRANSFORMER_KEY_DIM = 16        # TODO: try 8, 16, 32
TRANSFORMER_FF_DIM = 64         # TODO: try 32, 64, 128
TRANSFORMER_BLOCKS = 2          # TODO: try 1, 2, 3
TRANSFORMER_DROPOUT = 0.15      # TODO: try 0.05, 0.10, 0.20
TRANSFORMER_EPOCHS = EPOCHS


In [ ]:
class SinusoidalPositionEncoding(tf.keras.layers.Layer):
    """Fixed sinusoidal positional encoding for continuous time-series windows.

    This is different from an NLP token embedding. We first project continuous
    features with Dense(...), then add a deterministic position signal.
    """

    def __init__(self, sequence_length: int, model_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.model_dim = model_dim

    def call(self, inputs):
        positions = tf.cast(tf.range(self.sequence_length)[:, tf.newaxis], tf.float32)
        dims = tf.cast(tf.range(self.model_dim)[tf.newaxis, :], tf.float32)
        angle_rates = 1.0 / tf.pow(10000.0, (2.0 * tf.floor(dims / 2.0)) / tf.cast(self.model_dim, tf.float32))
        angle_rads = positions * angle_rates

        # Apply sin to even dimensions and cos to odd dimensions.
        even_mask = tf.cast(tf.math.equal(tf.math.mod(tf.range(self.model_dim), 2), 0), tf.float32)
        even_mask = even_mask[tf.newaxis, :]
        pos_encoding = tf.sin(angle_rads) * even_mask + tf.cos(angle_rads) * (1.0 - even_mask)
        pos_encoding = pos_encoding[tf.newaxis, :, :]
        return inputs + pos_encoding

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "model_dim": self.model_dim,
        })
        return config


def transformer_encoder_block(x, num_heads: int, key_dim: int, ff_dim: int, dropout: float):
    # Attention block.
    attn_input = LayerNormalization(epsilon=1e-6)(x)
    attn_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout
    )(attn_input, attn_input)
    x = Add()([x, attn_output])

    # Feed-forward block.
    ffn_input = LayerNormalization(epsilon=1e-6)(x)
    ffn_output = Dense(ff_dim, activation="relu")(ffn_input)
    ffn_output = Dropout(dropout)(ffn_output)
    ffn_output = Dense(x.shape[-1])(ffn_output)
    x = Add()([x, ffn_output])
    return x


def build_time_series_transformer(input_shape) -> tf.keras.Model:
    inputs = Input(shape=input_shape)

    # Project continuous features to the Transformer model dimension.
    x = Dense(TRANSFORMER_MODEL_DIM)(inputs)
    x = SinusoidalPositionEncoding(input_shape[0], TRANSFORMER_MODEL_DIM)(x)

    for _ in range(TRANSFORMER_BLOCKS):
        x = transformer_encoder_block(
            x,
            num_heads=TRANSFORMER_HEADS,
            key_dim=TRANSFORMER_KEY_DIM,
            ff_dim=TRANSFORMER_FF_DIM,
            dropout=TRANSFORMER_DROPOUT,
        )

    x = LayerNormalization(epsilon=1e-6)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(TRANSFORMER_DROPOUT)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(TRANSFORMER_DROPOUT)(x)
    outputs = Dense(1, activation=TARGET_CONFIG["output_activation"])(x)

    transformer = Model(inputs, outputs, name="TimeSeriesTransformer")
    transformer.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=TARGET_CONFIG["loss"],
        metrics=["mae"]
    )
    return transformer

transformer_model = build_time_series_transformer(input_shape=(X_train.shape[1], X_train.shape[2]))
transformer_model.summary()

## **29. Train the Time-Series Transformer**

This is the requested Transformer model added at the end of the notebook.


In [ ]:
print("Training the Time-Series Transformer...")

transformer_history = transformer_model.fit(
    X_train,
    y_train,
    epochs=TRANSFORMER_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1,
    shuffle=False,
    callbacks=callbacks,
)

plot_training_history(transformer_history, "TimeSeriesTransformer")


## **30. Evaluate the Transformer and Compare Models**

The final comparison table lets students decide which model performed better on their selected stock and settings.


In [ ]:
actual_target_transformer, transformer_pred, transformer_result = evaluate_sequence_model(
    transformer_model,
    "TimeSeriesTransformer"
)

transformer_prediction_df = pd.DataFrame({
    "Date": test_dates.values,
    f"Actual {DL_TARGET}": actual_target_transformer,
    f"Predicted {DL_TARGET}": transformer_pred,
})

fig = px.line(
    transformer_prediction_df,
    x="Date",
    y=[f"Actual {DL_TARGET}", f"Predicted {DL_TARGET}"],
    title=plot_title(f"Actual vs Predicted {DL_TARGET}", TICKER, "TimeSeriesTransformer + Plotly"),
    labels={"value": DL_TARGET, "variable": "Series"},
)
fig.update_layout(width=1200, height=600)
show_plotly(fig, image_name=f"{safe_ticker_name(TICKER)}_transformer_target_prediction.png")

plot_reconstructed_price_path(transformer_pred, "TimeSeriesTransformer")

results_df = pd.DataFrame(model_results).drop_duplicates(subset=["Model", "Target"], keep="last")
results_df = results_df.sort_values("RMSE").reset_index(drop=True)
print("Model comparison table:")
print(results_df)
print("\nKey question: Did the recurrent model or Transformer beat PersistenceBaseline with Theil's U < 1?")

## **31. Reflection Questions for Students**

Answer these after running your own ticker:

1. Which stock did you choose and why?
2. What date range did you use?
3. Why is global mean imputation invalid for a stock-price time series?
4. Did forward fill or past-only expanding mean make more sense for your missing-data example? Why?
5. What did the moving averages suggest about the descriptive trend?
6. Which days had unusually high trading volume? Did you find any possible explanation?
7. What did the ADF test suggest about raw `Close` price versus `Log_Return`?
8. Why did the linear regression section predict `Log_Return` instead of raw `Close`?
9. Which recurrent architecture did you choose: RNN, LSTM, GRU, BiRNN, or BiLSTM?
10. If you used `RNN` with a long window, what vanishing-gradient concern should you mention?
11. Did the Transformer perform better or worse than the recurrent model? Use MAE/RMSE/Theil's U evidence.
12. Did any complex model beat the persistence baseline? If not, what does that imply?
13. What did you change in the notebook to make it your own?

## **33. Submission Instructions**

1. Change `STUDENT_NAME`, `TICKER`, `COMPANY_NAME`, `SECOND_TICKER`, and `SECOND_COMPANY_NAME`.
2. Run the notebook from top to bottom.
3. Modify at least three student-task parameters, such as ticker, date range, moving-average window, recurrent model, Transformer heads, target variable, or number of epochs.
4. Make sure your graphs show your name and your chosen company/ticker.
5. Save this completed notebook.
6. Export or print the notebook to PDF.
7. Submit the notebook and PDF. The saved graph images will be in the Google Drive visualization folder when using Colab, or in `student_graphs` when running locally.
8. After finishing, upload the completed notebook and saved PNG figures to an LLM such as Gemini, ChatGPT, Claude, or another approved tool and ask it to create a **two-column IEEE-style project report** using the prompt in the next section.

**Final reminder:** Your work should be about your chosen company, not necessarily Microsoft. The report must use your actual metrics and figures. Do not let the LLM invent results.

## **Advanced ML: Apple and Microsoft Sequence-Model Benchmark**

This section adds the requested advanced machine-learning benchmark for **Apple (AAPL)** and **Microsoft (MSFT)**.

It trains and compares the following models on the same next-day log-return forecasting task:

- RNN
- LSTM
- GRU
- BiRNN
- BiLSTM
- Time-Series Transformer

The cell is self-contained: it downloads AAPL/MSFT data, builds causal financial features, trains all six models for both stocks, computes metrics, and saves/displays the comparison plots as PNG images. No HTML files are created.

In [ ]:

# ============================================================
# ADVANCED ML: APPLE + MICROSOFT SEQUENCE MODEL BENCHMARK
# Models: RNN, LSTM, GRU, BiRNN, BiLSTM, Time-Series Transformer
# Task: next-day log-return forecasting for AAPL and MSFT
# ============================================================

# ---- 0. Install dependencies if needed ----
import sys, subprocess, importlib, os, warnings, math, random
warnings.filterwarnings("ignore")

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("yfinance", "yfinance"),
    ("sklearn", "scikit-learn"),
    ("plotly", "plotly"),
    ("kaleido", "kaleido"),
]:
    ensure_package(import_name, pip_name)

# TensorFlow is usually preinstalled in Colab. Install only if missing.
ensure_package("tensorflow", "tensorflow")

# ---- 1. Imports ----
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from IPython.display import display, Image
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ---- 2. Output directory: Drive if available, otherwise local ----
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/Apple_Microsoft_Sequence_Model_Results"
except Exception:
    OUTPUT_DIR = "Apple_Microsoft_Sequence_Model_Results"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Saving figures/results to:", OUTPUT_DIR)

# ---- 3. Configuration ----
TICKERS = ["AAPL", "MSFT"]
START_DATE = "2018-01-01"
END_DATE = None              # None = latest available
WINDOW_SIZE = 30
TRAIN_RATIO = 0.80

EPOCHS = 12                  # increase for final runs
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
PATIENCE = 4

FEATURE_COLUMNS = [
    "Log_Return",
    "Volatility_20",
    "RSI_14",
    "Volume_Change",
    "MA_5_Ratio",
    "MA_20_Ratio",
]
TARGET_COLUMN = "Next_Log_Return"

MODEL_NAMES = ["RNN", "LSTM", "GRU", "BiRNN", "BiLSTM", "Transformer"]

# ---- 4. Utility functions ----
def safe_name(x):
    return "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in str(x))

def display_and_save_plotly(fig, filename, width=1100, height=650):
    """Save a Plotly figure as PNG and display it in the notebook. No HTML is created."""
    path = os.path.join(OUTPUT_DIR, filename)
    try:
        fig.write_image(path, width=width, height=height, scale=2)
        display(Image(filename=path))
        print("Saved:", path)
    except Exception as e:
        print("PNG export failed. Displaying interactive figure instead.")
        print("Reason:", repr(e))
        fig.show()
    return path

def compute_rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / (loss + 1e-12)
    return 100 - (100 / (1 + rs))

def download_and_engineer_features(ticker):
    df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
    if df.empty:
        raise ValueError(f"No data downloaded for {ticker}. Check ticker/date/internet connection.")

    # yfinance sometimes returns a MultiIndex.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    # Time-causal finance features.
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))
    df["Volatility_20"] = df["Log_Return"].rolling(20).std() * np.sqrt(252)
    df["RSI_14"] = compute_rsi(df["Close"], 14)
    df["Volume_Change"] = np.log((df["Volume"] + 1) / (df["Volume"].shift(1) + 1))
    df["MA_5"] = df["Close"].rolling(5).mean()
    df["MA_20"] = df["Close"].rolling(20).mean()
    df["MA_5_Ratio"] = df["Close"] / df["MA_5"] - 1.0
    df["MA_20_Ratio"] = df["Close"] / df["MA_20"] - 1.0

    # Predict next day's log return.
    df[TARGET_COLUMN] = df["Log_Return"].shift(-1)

    # Keep only valid rows.
    required = ["Date", "Close", "Volume"] + FEATURE_COLUMNS + [TARGET_COLUMN]
    df = df[required].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return df

def make_windows(df, feature_cols, target_col, window_size):
    X, y, dates, close_anchor, close_next = [], [], [], [], []
    features = df[feature_cols].values.astype(np.float32)
    target = df[target_col].values.astype(np.float32)
    close = df["Close"].values.astype(np.float32)
    date_values = df["Date"].values

    # Window uses rows [i, ..., i+window-1] to predict target at row i+window-1,
    # which is the next-day return from close[i+window-1] to close[i+window].
    for i in range(len(df) - window_size):
        end = i + window_size - 1
        X.append(features[i : i + window_size])
        y.append(target[end])
        dates.append(pd.to_datetime(date_values[end]))
        close_anchor.append(close[end])
        close_next.append(close[end + 1] if end + 1 < len(close) else np.nan)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32).reshape(-1, 1),
        pd.to_datetime(dates),
        np.asarray(close_anchor, dtype=np.float32).reshape(-1, 1),
        np.asarray(close_next, dtype=np.float32).reshape(-1, 1),
    )

def train_test_scale(X, y, train_ratio):
    n = len(X)
    split = int(train_ratio * n)

    X_train_raw, X_test_raw = X[:split], X[split:]
    y_train_raw, y_test_raw = y[:split], y[split:]

    # Fit scalers on training data only.
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    n_train, t_steps, n_feat = X_train_raw.shape
    X_train_2d = X_train_raw.reshape(-1, n_feat)
    X_test_2d = X_test_raw.reshape(-1, n_feat)

    X_train = x_scaler.fit_transform(X_train_2d).reshape(n_train, t_steps, n_feat)
    X_test = x_scaler.transform(X_test_2d).reshape(X_test_raw.shape[0], t_steps, n_feat)

    y_train = y_scaler.fit_transform(y_train_raw)
    y_test = y_scaler.transform(y_test_raw)

    return X_train, X_test, y_train, y_test, y_train_raw, y_test_raw, x_scaler, y_scaler, split

# ---- 5. Model definitions ----
def build_recurrent_model(model_name, input_shape):
    model = models.Sequential(name=model_name)

    if model_name == "RNN":
        model.add(layers.SimpleRNN(64, input_shape=input_shape))
    elif model_name == "LSTM":
        model.add(layers.LSTM(64, input_shape=input_shape))
    elif model_name == "GRU":
        model.add(layers.GRU(64, input_shape=input_shape))
    elif model_name == "BiRNN":
        model.add(layers.Bidirectional(layers.SimpleRNN(64), input_shape=input_shape))
    elif model_name == "BiLSTM":
        model.add(layers.Bidirectional(layers.LSTM(64), input_shape=input_shape))
    else:
        raise ValueError(f"Unknown recurrent model: {model_name}")

    model.add(layers.Dropout(0.20))
    model.add(layers.Dense(32, activation="relu"))
    model.add(layers.Dense(1, activation="linear"))

    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=["mae"],
    )
    return model

class SinusoidalPositionEncoding(layers.Layer):
    """Fixed sinusoidal positional encoding for continuous time-series inputs."""
    def call(self, x):
        seq_len = tf.shape(x)[1]
        d_model = tf.shape(x)[2]

        positions = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
        dims = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)

        angle_rates = 1.0 / tf.pow(
            10000.0,
            (2.0 * tf.floor(dims / 2.0)) / tf.cast(d_model, tf.float32)
        )
        angles = positions * angle_rates

        sin_mask = tf.cast(tf.math.floormod(tf.range(d_model), 2) == 0, tf.float32)
        cos_mask = 1.0 - sin_mask

        pos_encoding = tf.sin(angles) * sin_mask + tf.cos(angles) * cos_mask
        pos_encoding = pos_encoding[tf.newaxis, :, :]

        return x + pos_encoding

def build_transformer_model(input_shape, d_model=64, num_heads=4, ff_dim=128, dropout=0.15):
    inp = layers.Input(shape=input_shape)

    # Continuous projection: correct for numerical time-series data.
    x = layers.Dense(d_model)(inp)
    x = SinusoidalPositionEncoding()(x)

    # Transformer block 1
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff)

    # Transformer block 2
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(1, activation="linear")(x)

    model = models.Model(inp, out, name="TimeSeriesTransformer")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=["mae"],
    )
    return model

def build_model(model_name, input_shape):
    if model_name == "Transformer":
        return build_transformer_model(input_shape)
    return build_recurrent_model(model_name, input_shape)

# ---- 6. Evaluation helpers ----
def directional_accuracy(y_true, y_pred):
    return float(np.mean(np.sign(y_true.reshape(-1)) == np.sign(y_pred.reshape(-1))))

def metrics_dict(y_true, y_pred, y_naive):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    rmse_naive = float(np.sqrt(mean_squared_error(y_true, y_naive)))
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse,
        "R2": float(r2_score(y_true, y_pred)),
        "Theils_U_vs_Persistence": rmse / (rmse_naive + 1e-12),
        "Directional_Accuracy": directional_accuracy(y_true, y_pred),
    }

def fit_one_model(model_name, X_train, y_train, X_test, y_test, y_scaler):
    tf.keras.backend.clear_session()
    model = build_model(model_name, input_shape=X_train.shape[1:])

    es = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )

    hist = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        verbose=0,
        callbacks=[es],
    )

    pred_scaled = model.predict(X_test, verbose=0)
    pred = y_scaler.inverse_transform(pred_scaled)
    return model, hist, pred

# ---- 7. Run benchmark for AAPL and MSFT ----
all_results = []
all_predictions = {}

for ticker in TICKERS:
    print("\n" + "=" * 80)
    print(f"Processing {ticker}")
    print("=" * 80)

    df = download_and_engineer_features(ticker)
    X, y, dates, close_anchor, close_next = make_windows(df, FEATURE_COLUMNS, TARGET_COLUMN, WINDOW_SIZE)

    X_train, X_test, y_train, y_test, y_train_raw, y_test_raw, x_scaler, y_scaler, split = train_test_scale(X, y, TRAIN_RATIO)

    test_dates = dates[split:]
    test_close_anchor = close_anchor[split:]
    test_close_next = close_next[split:]

    # Naive persistence baseline: next log return = most recent observed log return in window.
    # The last feature column index for Log_Return is 0 in FEATURE_COLUMNS.
    naive_persistence = X[split:, -1, FEATURE_COLUMNS.index("Log_Return")].reshape(-1, 1)
    naive_zero = np.zeros_like(y_test_raw)

    baseline_metrics = metrics_dict(y_test_raw, naive_persistence, naive_persistence)
    baseline_metrics.update({"Ticker": ticker, "Model": "NaivePersistence"})
    all_results.append(baseline_metrics)

    zero_metrics = metrics_dict(y_test_raw, naive_zero, naive_persistence)
    zero_metrics.update({"Ticker": ticker, "Model": "ZeroReturn"})
    all_results.append(zero_metrics)

    all_predictions[(ticker, "Actual")] = {
        "dates": test_dates,
        "y_true": y_test_raw,
        "y_pred": y_test_raw,
        "close_anchor": test_close_anchor,
        "close_next": test_close_next,
    }
    all_predictions[(ticker, "NaivePersistence")] = {
        "dates": test_dates,
        "y_true": y_test_raw,
        "y_pred": naive_persistence,
        "close_anchor": test_close_anchor,
        "close_next": test_close_next,
    }

    for model_name in MODEL_NAMES:
        print(f"Training {ticker} - {model_name} ...")
        model, hist, pred = fit_one_model(model_name, X_train, y_train, X_test, y_test, y_scaler)

        m = metrics_dict(y_test_raw, pred, naive_persistence)
        m.update({
            "Ticker": ticker,
            "Model": model_name,
            "Epochs_Run": len(hist.history["loss"]),
            "Final_Train_Loss": float(hist.history["loss"][-1]),
            "Final_Val_Loss": float(hist.history["val_loss"][-1]),
        })
        all_results.append(m)

        all_predictions[(ticker, model_name)] = {
            "dates": test_dates,
            "y_true": y_test_raw,
            "y_pred": pred,
            "close_anchor": test_close_anchor,
            "close_next": test_close_next,
            "history": hist.history,
        }

results_df = pd.DataFrame(all_results)
results_df = results_df[[
    "Ticker", "Model", "MAE", "RMSE", "R2", "Theils_U_vs_Persistence",
    "Directional_Accuracy"
] + [c for c in ["Epochs_Run", "Final_Train_Loss", "Final_Val_Loss"] if c in results_df.columns]]

csv_path = os.path.join(OUTPUT_DIR, "apple_microsoft_model_metrics.csv")
results_df.to_csv(csv_path, index=False)
display(results_df.sort_values(["Ticker", "RMSE"]))
print("Saved metrics:", csv_path)

# ---- 8. Visualizations: combined Apple/Microsoft results ----

# 8.1 RMSE comparison
fig = go.Figure()
for ticker in TICKERS:
    sub = results_df[(results_df["Ticker"] == ticker) & (~results_df["Model"].isin(["NaivePersistence", "ZeroReturn"]))]
    fig.add_trace(go.Bar(
        x=sub["Model"],
        y=sub["RMSE"],
        name=ticker,
        hovertemplate="Ticker=%{fullData.name}<br>Model=%{x}<br>RMSE=%{y:.6f}<extra></extra>"
    ))
fig.update_layout(
    title="AAPL vs MSFT: RMSE by Sequence Model",
    xaxis_title="Model",
    yaxis_title="RMSE on next-day log return",
    barmode="group",
    width=1100,
    height=650,
)
display_and_save_plotly(fig, "aapl_msft_rmse_by_model.png")

# 8.2 Theil's U comparison
fig = go.Figure()
for ticker in TICKERS:
    sub = results_df[(results_df["Ticker"] == ticker) & (~results_df["Model"].isin(["NaivePersistence", "ZeroReturn"]))]
    fig.add_trace(go.Bar(
        x=sub["Model"],
        y=sub["Theils_U_vs_Persistence"],
        name=ticker,
        hovertemplate="Ticker=%{fullData.name}<br>Model=%{x}<br>Theil's U=%{y:.4f}<extra></extra>"
    ))
fig.add_hline(y=1.0, line_dash="dash", annotation_text="Persistence baseline")
fig.update_layout(
    title="AAPL vs MSFT: Theil's U vs Persistence Baseline",
    xaxis_title="Model",
    yaxis_title="Theil's U; below 1.0 beats persistence",
    barmode="group",
    width=1100,
    height=650,
)
display_and_save_plotly(fig, "aapl_msft_theils_u_by_model.png")

# 8.3 Directional accuracy comparison
fig = go.Figure()
for ticker in TICKERS:
    sub = results_df[(results_df["Ticker"] == ticker) & (~results_df["Model"].isin(["NaivePersistence", "ZeroReturn"]))]
    fig.add_trace(go.Bar(
        x=sub["Model"],
        y=sub["Directional_Accuracy"],
        name=ticker,
        hovertemplate="Ticker=%{fullData.name}<br>Model=%{x}<br>Directional Accuracy=%{y:.3f}<extra></extra>"
    ))
fig.add_hline(y=0.5, line_dash="dash", annotation_text="Random sign benchmark")
fig.update_layout(
    title="AAPL vs MSFT: Directional Accuracy",
    xaxis_title="Model",
    yaxis_title="Directional Accuracy",
    barmode="group",
    width=1100,
    height=650,
)
display_and_save_plotly(fig, "aapl_msft_directional_accuracy_by_model.png")

# 8.4 Actual vs predicted log returns, all models, both tickers.
fig = go.Figure()
for ticker in TICKERS:
    actual = all_predictions[(ticker, "Actual")]
    fig.add_trace(go.Scatter(
        x=actual["dates"],
        y=actual["y_true"].reshape(-1),
        mode="lines",
        name=f"{ticker} Actual",
        line=dict(width=3),
    ))
    for model_name in MODEL_NAMES:
        item = all_predictions[(ticker, model_name)]
        fig.add_trace(go.Scatter(
            x=item["dates"],
            y=item["y_pred"].reshape(-1),
            mode="lines",
            name=f"{ticker} {model_name}",
            opacity=0.65,
        ))
fig.update_layout(
    title="AAPL and MSFT: Actual vs Predicted Next-Day Log Returns",
    xaxis_title="Date",
    yaxis_title="Log return",
    width=1300,
    height=750,
)
display_and_save_plotly(fig, "aapl_msft_actual_vs_predicted_log_returns.png", width=1300, height=750)

# 8.5 Reconstructed one-step-ahead price paths from predicted log returns.
# predicted next close = anchor close * exp(predicted next log return)
fig = go.Figure()
for ticker in TICKERS:
    actual = all_predictions[(ticker, "Actual")]
    actual_next_close = actual["close_next"].reshape(-1)

    fig.add_trace(go.Scatter(
        x=actual["dates"],
        y=actual_next_close,
        mode="lines",
        name=f"{ticker} Actual Next Close",
        line=dict(width=3),
    ))

    for model_name in MODEL_NAMES:
        item = all_predictions[(ticker, model_name)]
        pred_next_close = item["close_anchor"].reshape(-1) * np.exp(item["y_pred"].reshape(-1))
        fig.add_trace(go.Scatter(
            x=item["dates"],
            y=pred_next_close,
            mode="lines",
            name=f"{ticker} {model_name}",
            opacity=0.65,
        ))

fig.update_layout(
    title="AAPL and MSFT: One-Step-Ahead Reconstructed Price Predictions",
    xaxis_title="Date",
    yaxis_title="Adjusted close price",
    width=1300,
    height=750,
)
display_and_save_plotly(fig, "aapl_msft_reconstructed_price_predictions.png", width=1300, height=750)

# 8.6 2D analog of the old 3D comparison: actual next close vs predicted next close.
# Each point compares the realized next close with the model-predicted next close.
# The dashed y=x reference line indicates perfect prediction.
fig = go.Figure()
all_actual_prices = []
all_pred_prices = []

for ticker in TICKERS:
    for model_name in MODEL_NAMES:
        item = all_predictions[(ticker, model_name)]
        actual_next_close = item["close_next"].reshape(-1)
        pred_next_close = item["close_anchor"].reshape(-1) * np.exp(item["y_pred"].reshape(-1))
        all_actual_prices.extend(actual_next_close.tolist())
        all_pred_prices.extend(pred_next_close.tolist())
        fig.add_trace(go.Scatter(
            x=actual_next_close,
            y=pred_next_close,
            mode="markers",
            name=f"{ticker} {model_name}",
            opacity=0.55,
            marker=dict(size=5),
            hovertemplate=(
                f"Ticker={ticker}<br>Model={model_name}"
                "<br>Actual next close=%{x:.2f}"
                "<br>Predicted next close=%{y:.2f}<extra></extra>"
            ),
        ))

min_price = float(np.nanmin([np.nanmin(all_actual_prices), np.nanmin(all_pred_prices)]))
max_price = float(np.nanmax([np.nanmax(all_actual_prices), np.nanmax(all_pred_prices)]))
fig.add_trace(go.Scatter(
    x=[min_price, max_price],
    y=[min_price, max_price],
    mode="lines",
    name="Perfect prediction line",
    line=dict(dash="dash", width=3),
    hoverinfo="skip",
))

fig.update_layout(
    title="AAPL and MSFT: 2D Actual-vs-Predicted Next-Close Comparison",
    xaxis_title="Actual next close",
    yaxis_title="Predicted next close",
    width=1100,
    height=750,
)
display_and_save_plotly(fig, "aapl_msft_2d_actual_vs_predicted_price_scatter.png", width=1100, height=750)

# 8.7 2D prediction-error-over-time analog.
# This shows where each model over-predicts or under-predicts across the test period.
fig = go.Figure()
for ticker in TICKERS:
    for model_name in MODEL_NAMES:
        item = all_predictions[(ticker, model_name)]
        actual_next_close = item["close_next"].reshape(-1)
        pred_next_close = item["close_anchor"].reshape(-1) * np.exp(item["y_pred"].reshape(-1))
        error = pred_next_close - actual_next_close
        fig.add_trace(go.Scatter(
            x=item["dates"],
            y=error,
            mode="lines",
            name=f"{ticker} {model_name}",
            opacity=0.70,
            hovertemplate=(
                f"Ticker={ticker}<br>Model={model_name}"
                "<br>Date=%{x}"
                "<br>Prediction error=%{y:.3f}<extra></extra>"
            ),
        ))
fig.add_hline(y=0.0, line_dash="dash", annotation_text="zero error")
fig.update_layout(
    title="AAPL and MSFT: 2D Prediction Error Over Time",
    xaxis_title="Date",
    yaxis_title="Predicted next close - Actual next close",
    width=1300,
    height=750,
)
display_and_save_plotly(fig, "aapl_msft_2d_prediction_error_over_time.png", width=1300, height=750)

print("\nCompleted Apple/Microsoft benchmark for:")
print(", ".join(MODEL_NAMES))
print("Figures and metrics are in:", OUTPUT_DIR)


## **34. LLM-Assisted IEEE Two-Column Report Builder**

After you finish the notebook, use the following cell to generate a structured prompt and project summary. Then upload your completed notebook and saved figures to an LLM such as Gemini and paste the generated prompt.

The LLM should produce a **two-column IEEE-style report**, not a casual summary. The report should include exact metrics from your notebook, mention whether your deep models beat the naive baseline, and avoid unsupported claims.

Also include the **Advanced ML** results comparing RNN, LSTM, GRU, BiRNN, BiLSTM, and the Time-Series Transformer for AAPL and MSFT if you ran that section.

In [ ]:
# Generate a report prompt that students can paste into Gemini, ChatGPT, Claude, or another approved LLM.
# Run this after all model cells have completed.
# Recommended workflow:
#   1. Finish the notebook.
#   2. Download the completed notebook and use the saved PNG figures from the Drive visualization folder.
#   3. Give the notebook/results to Gemini or another approved LLM.
#   4. Ask it to write a two-column IEEE-style report using ONLY your real outputs.

from pathlib import Path

report_path = Path(OUTPUT_DIR) / f"{safe_ticker_name(TICKER)}_ieee_report_prompt.md"
figure_files = []
if Path(OUTPUT_DIR).exists():
    figure_files = sorted(set(p.name for p in Path(OUTPUT_DIR).glob("*.png")))

results_text = "Deep-learning results were not generated yet. Run the recurrent and Transformer sections first."
if "results_df" in globals():
    results_text = results_df.to_string(index=False)

ml_text = "Linear-regression metrics were not generated yet. Run Section 13 first."
if "ml_metrics" in globals():
    ml_text = ml_metrics.to_string(index=False)

ols_text = "OLS metrics were not generated yet. Run Section 12 first."
if all(name in globals() for name in ["ols_mae", "ols_rmse", "ols_zero_rmse", "ols_persist_rmse"]):
    ols_text = (
        f"OLS MAE={ols_mae:.8f}, OLS RMSE={ols_rmse:.8f}, "
        f"Zero-return RMSE={ols_zero_rmse:.8f}, Persistence RMSE={ols_persist_rmse:.8f}, "
        f"Theil's U vs persistence={ols_rmse / ols_persist_rmse:.4f}"
    )

adf_text = "ADF tests were skipped or not generated."
if "adf_close" in globals() and "adf_return" in globals():
    adf_text = (
        f"ADF raw Close p-value={adf_close['p_value']:.6f}; "
        f"ADF Log_Return p-value={adf_return['p_value']:.6f}."
    )

report_prompt = f"""
You are an academic writing assistant. Create a two-column IEEE-style project report from my completed Jupyter notebook.

STRICT RULES:
1. Use only the results, figures, settings, and metrics from my notebook. Do not invent numbers.
2. Write in formal academic style.
3. Use IEEE-style section names and concise technical language.
4. Include a clear statement that this is an educational forecasting experiment, not investment advice.
5. Discuss methodological safeguards: no global mean imputation, sequential train/test split, lagged predictors, return-based regression, naive baselines, Theil's U, target-specific deep-learning output layers, and sinusoidal Transformer positional encoding.
6. Mention that most visualizations were built with Plotly, saved as PNG files to the visualization output folder, and displayed in the notebook from the saved image files. No HTML figure files were used.
7. Format the report as a two-column IEEE paper draft. Use sections, figure captions, and tables.

PROJECT SETTINGS:
- Student/Analyst: {STUDENT_NAME}
- Main ticker: {TICKER}
- Company: {COMPANY_NAME}
- Second ticker: {SECOND_TICKER}
- Second company: {SECOND_COMPANY_NAME}
- Date range: {START_DATE} to {END_DATE}
- Train ratio: {TRAIN_RATIO}
- Window size: {WINDOW_SIZE}
- Deep-learning target: {DL_TARGET}
- Chosen recurrent model: {CHOSEN_RECURRENT_MODEL}
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}

STATIONARITY EVIDENCE:
{adf_text}

STATSMODELS OLS RETURN-FORECAST SUMMARY:
{ols_text}

SCIKIT-LEARN LINEAR REGRESSION AND BASELINES:
{ml_text}

DEEP-LEARNING MODEL AND BASELINE RESULTS:
{results_text}

AVAILABLE FIGURE FILES:
{chr(10).join(figure_files) if figure_files else 'No saved figures found yet.'}

REQUIRED IEEE-STYLE REPORT STRUCTURE:
- Title
- Abstract
- I. Introduction
- II. Dataset and Feature Engineering
- III. Methodology
  - Missing-data handling without look-ahead bias
  - Stationarity and return-based modeling
  - Linear regression baseline
  - Recurrent neural networks
  - Time-series Transformer encoder
  - Naive baselines and Theil's U
  - Plotly visual analytics saved as PNG files, including two-stock comparison plots
- IV. Experimental Results
  - Include at least one table comparing Linear Regression, PersistenceBaseline, ZeroReturnBaseline if available, RNN/LSTM/GRU/BiLSTM if run, and Transformer.
  - Mention whether each complex model beat the persistence baseline.
- V. Discussion and Limitations
- VI. Conclusion
- References

IMPORTANT INTERPRETATION REQUIREMENT:
If Theil's U is greater than or equal to 1, state that the complex model did not outperform the naive persistence baseline. Do not claim predictive alpha unless the notebook evidence supports it.

DELIVERABLE:
Write the report in LaTeX-compatible IEEE style. Use placeholders such as \\includegraphics{{...}} for figure files that I should insert manually.
""".strip()

report_path.write_text(report_prompt, encoding="utf-8")
print(report_prompt)
print(f"\nSaved LLM report prompt to: {report_path}")

## Advanced ML: Two-Month Stock Forecast from Latest Data

This final single cell trains RNN, LSTM, GRU, BiRNN, BiLSTM, and a Time-Series Transformer for Apple and Microsoft. It plots one real historical close-price line and then extends forecasts from the latest available market date for approximately two trading months. The output is 2D only and saves PNG figures plus CSV tables.

In [ ]:

# ================================================================
# ADVANCED ML: TWO-MONTH STOCK FORECAST FROM THE LATEST DATA POINT
# Models: RNN, LSTM, GRU, BiRNN, BiLSTM, Time-Series Transformer
# Stocks: Apple (AAPL) and Microsoft (MSFT)
# Output: one real historical price line + forecast lines from all models
# ================================================================

# This cell is intentionally self-contained. It can be run at the end of the notebook.
# It forecasts approximately two trading months ahead using recursive next-day log-return prediction.

# -------------------------------
# 0. Install/import dependencies
# -------------------------------
import sys, subprocess, os, warnings, math, random
warnings.filterwarnings("ignore")

REQUIRED_PACKAGES = ["yfinance", "plotly", "kaleido", "tensorflow", "scikit-learn"]
for pkg in REQUIRED_PACKAGES:
    try:
        __import__(pkg if pkg != "scikit-learn" else "sklearn")
    except Exception:
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import yfinance as yf
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import plotly.graph_objects as go
from IPython.display import display, Image as IPyImage

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# -------------------------------
# 1. User controls
# -------------------------------
FORECAST_TICKERS = ["AAPL", "MSFT"]
FORECAST_START_DATE = "2018-01-01"
FORECAST_HORIZON_DAYS = 42          # about two trading months
LOOKBACK_DAYS = 30                  # sequence length
FORECAST_EPOCHS = 10                # increase to 20-50 for stronger training
FORECAST_BATCH_SIZE = 32
VALIDATION_FRACTION = 0.15
PLOT_HISTORY_DAYS = 252             # show last trading year before forecast
CLIP_DAILY_LOG_RETURN = 0.10        # safety cap for recursive forecasts

MODEL_NAMES = ["RNN", "LSTM", "GRU", "BiRNN", "BiLSTM", "Time-Series Transformer"]
FEATURE_COLS = ["Log_Return", "Volatility_20", "RSI_14", "Momentum_5", "MA10_Gap"]
TARGET_COL = "Log_Return"

# Save to Google Drive if available; otherwise save locally.
AUTO_MOUNT_DRIVE = False  # set True in Colab if you want to mount Drive from this cell
try:
    if AUTO_MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if os.path.isdir("/content/drive/MyDrive"):
        OUTPUT_DIR = "/content/drive/MyDrive/CPS3320_Advanced_ML_Two_Month_Forecasts"
    else:
        OUTPUT_DIR = "CPS3320_Advanced_ML_Two_Month_Forecasts"
except Exception:
    OUTPUT_DIR = "CPS3320_Advanced_ML_Two_Month_Forecasts"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Saving forecast outputs to:", os.path.abspath(OUTPUT_DIR))

# -------------------------------
# 2. Feature engineering
# -------------------------------
def flatten_yfinance_columns(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    return data


def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / (loss + 1e-12)
    return 100 - (100 / (1 + rs))


def make_causal_features(price_df: pd.DataFrame) -> pd.DataFrame:
    """Create features using only current/past close values."""
    df = price_df.copy()
    df = df[["Close"]].dropna()
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))
    df["Volatility_20"] = df["Log_Return"].rolling(20).std() * np.sqrt(252)
    df["RSI_14"] = compute_rsi(df["Close"], 14)
    df["Momentum_5"] = np.log(df["Close"] / df["Close"].shift(5))
    df["MA10_Gap"] = (df["Close"] / df["Close"].rolling(10).mean()) - 1.0
    return df.dropna()


def create_sequences(feature_array: np.ndarray, target_array: np.ndarray, lookback: int):
    X, y = [], []
    for i in range(len(feature_array) - lookback):
        X.append(feature_array[i:i + lookback])
        y.append(target_array[i + lookback])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

# -------------------------------
# 3. Model definitions
# -------------------------------
class SinusoidalPositionEncoding(layers.Layer):
    """Fixed sinusoidal positional encoding for continuous time-series inputs."""
    def __init__(self, sequence_length: int, model_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.model_dim = model_dim
        position = np.arange(sequence_length)[:, np.newaxis]
        div_term = np.exp(np.arange(0, model_dim, 2) * -(np.log(10000.0) / model_dim))
        pe = np.zeros((sequence_length, model_dim), dtype=np.float32)
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.pe = tf.constant(pe[np.newaxis, :, :], dtype=tf.float32)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

    def get_config(self):
        config = super().get_config()
        config.update({"sequence_length": self.sequence_length, "model_dim": self.model_dim})
        return config


def build_sequence_model(model_name: str, input_shape):
    n_steps, n_features = input_shape
    inputs = layers.Input(shape=input_shape)

    if model_name == "RNN":
        x = layers.SimpleRNN(64, return_sequences=False)(inputs)
    elif model_name == "LSTM":
        x = layers.LSTM(64, return_sequences=False)(inputs)
    elif model_name == "GRU":
        x = layers.GRU(64, return_sequences=False)(inputs)
    elif model_name == "BiRNN":
        x = layers.Bidirectional(layers.SimpleRNN(64, return_sequences=False))(inputs)
    elif model_name == "BiLSTM":
        x = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(inputs)
    elif model_name == "Time-Series Transformer":
        model_dim = 64
        x = layers.Dense(model_dim)(inputs)  # continuous projection, not token embedding
        x = SinusoidalPositionEncoding(n_steps, model_dim)(x)
        attn = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.10)(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
        ffn = layers.Dense(128, activation="relu")(x)
        ffn = layers.Dropout(0.10)(ffn)
        ffn = layers.Dense(model_dim)(ffn)
        x = layers.LayerNormalization(epsilon=1e-6)(x + ffn)
        x = layers.GlobalAveragePooling1D()(x)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    x = layers.Dropout(0.20)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="linear")(x)  # target is next-day log return
    model = Model(inputs, outputs, name=model_name.replace(" ", "_"))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=tf.keras.losses.Huber())
    return model

# -------------------------------
# 4. Forecasting utilities
# -------------------------------
def recursive_price_forecast(model, raw_price_df, x_scaler, y_scaler, future_dates):
    """Forecast future prices by recursively predicting next-day log returns."""
    simulated = raw_price_df[["Close"]].copy().dropna()
    forecasts = []

    for future_date in future_dates:
        feat_df = make_causal_features(simulated)
        last_window = feat_df[FEATURE_COLS].tail(LOOKBACK_DAYS)
        if len(last_window) < LOOKBACK_DAYS:
            raise ValueError("Not enough rows to create the final forecast window.")

        x_scaled = x_scaler.transform(last_window)
        x_input = x_scaled.reshape(1, LOOKBACK_DAYS, len(FEATURE_COLS)).astype(np.float32)
        pred_scaled = model.predict(x_input, verbose=0)
        pred_log_return = float(y_scaler.inverse_transform(pred_scaled)[0, 0])
        pred_log_return = float(np.clip(pred_log_return, -CLIP_DAILY_LOG_RETURN, CLIP_DAILY_LOG_RETURN))

        last_close = float(simulated["Close"].iloc[-1])
        next_close = last_close * math.exp(pred_log_return)
        simulated.loc[pd.Timestamp(future_date), "Close"] = next_close
        forecasts.append({
            "Date": pd.Timestamp(future_date),
            "Predicted_Log_Return": pred_log_return,
            "Predicted_Close": next_close
        })

    return pd.DataFrame(forecasts)


def safe_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in str(text))


def save_plotly_png(fig, filename: str):
    path = os.path.join(OUTPUT_DIR, filename)
    try:
        fig.write_image(path, scale=2, width=1200, height=700)
        display(IPyImage(filename=path))
        return path
    except Exception as e:
        print("PNG export through Plotly/Kaleido failed; showing interactive figure instead.")
        print("Reason:", repr(e))
        fig.show()
        return None

# -------------------------------
# 5. Train models and forecast AAPL/MSFT two months ahead
# -------------------------------
all_metrics = []
all_forecasts = []

for ticker in FORECAST_TICKERS:
    print("\n" + "=" * 80)
    print(f"Downloading and training forecast models for {ticker}")
    print("=" * 80)

    raw = yf.download(ticker, start=FORECAST_START_DATE, progress=False, auto_adjust=False)
    raw = flatten_yfinance_columns(raw)
    if raw.empty or "Close" not in raw.columns:
        raise RuntimeError(f"No usable Close data downloaded for {ticker}.")
    raw = raw[["Close"]].dropna()
    raw.index = pd.to_datetime(raw.index)

    feature_df = make_causal_features(raw)
    x_scaler = MinMaxScaler()
    y_scaler = MinMaxScaler()
    X_scaled_base = x_scaler.fit_transform(feature_df[FEATURE_COLS])
    y_scaled_base = y_scaler.fit_transform(feature_df[[TARGET_COL]])
    X_all, y_all = create_sequences(X_scaled_base, y_scaled_base, LOOKBACK_DAYS)

    split_idx = int((1.0 - VALIDATION_FRACTION) * len(X_all))
    X_train, X_val = X_all[:split_idx], X_all[split_idx:]
    y_train, y_val = y_all[:split_idx], y_all[split_idx:]

    latest_date = raw.index[-1]
    latest_close = float(raw["Close"].iloc[-1])
    future_dates = pd.bdate_range(latest_date + pd.offsets.BDay(1), periods=FORECAST_HORIZON_DAYS)

    ticker_forecasts = {}

    for model_name in MODEL_NAMES:
        print(f"Training {ticker} | {model_name}")
        tf.keras.backend.clear_session()
        model = build_sequence_model(model_name, input_shape=(LOOKBACK_DAYS, len(FEATURE_COLS)))
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=FORECAST_EPOCHS,
            batch_size=FORECAST_BATCH_SIZE,
            shuffle=False,
            verbose=0,
            callbacks=[early_stop]
        )

        val_pred_scaled = model.predict(X_val, verbose=0)
        val_pred = y_scaler.inverse_transform(val_pred_scaled).ravel()
        val_true = y_scaler.inverse_transform(y_val).ravel()
        naive_pred = np.zeros_like(val_true)  # zero-return baseline for next-day log return

        rmse = float(np.sqrt(mean_squared_error(val_true, val_pred)))
        mae = float(mean_absolute_error(val_true, val_pred))
        r2 = float(r2_score(val_true, val_pred))
        naive_rmse = float(np.sqrt(mean_squared_error(val_true, naive_pred)))
        theils_u = float(rmse / naive_rmse) if naive_rmse > 0 else np.nan
        directional_accuracy = float(np.mean(np.sign(val_pred) == np.sign(val_true)))

        forecast_df = recursive_price_forecast(model, raw, x_scaler, y_scaler, future_dates)
        forecast_df["Ticker"] = ticker
        forecast_df["Model"] = model_name
        forecast_df["Latest_Date"] = latest_date
        forecast_df["Latest_Close"] = latest_close
        ticker_forecasts[model_name] = forecast_df
        all_forecasts.append(forecast_df)

        all_metrics.append({
            "Ticker": ticker,
            "Model": model_name,
            "Validation_MAE_LogReturn": mae,
            "Validation_RMSE_LogReturn": rmse,
            "Validation_R2_LogReturn": r2,
            "Validation_Theils_U_vs_ZeroReturn": theils_u,
            "Validation_Directional_Accuracy": directional_accuracy,
            "Latest_Date": latest_date,
            "Latest_Close": latest_close,
            "Forecast_Horizon_Trading_Days": FORECAST_HORIZON_DAYS,
            "Forecast_Final_Close": float(forecast_df["Predicted_Close"].iloc[-1]),
            "Forecast_Total_Return": float((forecast_df["Predicted_Close"].iloc[-1] / latest_close) - 1.0)
        })

    # Add a flat naive line on the plot for visual reference.
    naive_flat = pd.DataFrame({
        "Date": future_dates,
        "Predicted_Close": [latest_close] * len(future_dates),
        "Predicted_Log_Return": [0.0] * len(future_dates),
        "Ticker": ticker,
        "Model": "Naive Flat Baseline",
        "Latest_Date": latest_date,
        "Latest_Close": latest_close
    })
    ticker_forecasts["Naive Flat Baseline"] = naive_flat
    all_forecasts.append(naive_flat)

    # -------------------------------
    # 6. Plot: one real line + all model forecasts from latest date
    # -------------------------------
    hist = raw.tail(PLOT_HISTORY_DAYS).copy()
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist["Close"],
        mode="lines",
        name=f"{ticker} real Close",
        line=dict(width=3, color="black")
    ))

    for model_name, fdf in ticker_forecasts.items():
        joined_x = [latest_date] + list(fdf["Date"])
        joined_y = [latest_close] + list(fdf["Predicted_Close"])
        line_style = "dash" if "Naive" in model_name else "solid"
        fig.add_trace(go.Scatter(
            x=joined_x,
            y=joined_y,
            mode="lines",
            name=model_name,
            line=dict(width=2, dash=line_style)
        ))

    fig.add_vline(x=latest_date, line_width=2, line_dash="dot", line_color="gray")
    fig.update_layout(
        title=f"{ticker}: Real Close Price and 2-Month Forecasts from All Sequence Models",
        xaxis_title="Date",
        yaxis_title="Close Price (USD)",
        legend_title="Series",
        template="plotly_white",
        width=1200,
        height=700
    )
    save_plotly_png(fig, f"{safe_name(ticker)}_two_month_all_model_forecast.png")

# -------------------------------
# 7. Save tables and combined normalized 2D comparison
# -------------------------------
metrics_df = pd.DataFrame(all_metrics).sort_values(["Ticker", "Validation_RMSE_LogReturn"])
forecasts_df = pd.concat(all_forecasts, ignore_index=True)

metrics_path = os.path.join(OUTPUT_DIR, "two_month_forecast_model_metrics.csv")
forecasts_path = os.path.join(OUTPUT_DIR, "two_month_forecast_paths.csv")
metrics_df.to_csv(metrics_path, index=False)
forecasts_df.to_csv(forecasts_path, index=False)

print("\nSaved metrics:", metrics_path)
print("Saved forecast paths:", forecasts_path)
display(metrics_df)

# Combined 2D normalized plot for AAPL and MSFT: actual recent history + future forecasts.
combined_fig = go.Figure()
for ticker in FORECAST_TICKERS:
    raw = yf.download(ticker, start=FORECAST_START_DATE, progress=False, auto_adjust=False)
    raw = flatten_yfinance_columns(raw)[["Close"]].dropna()
    raw.index = pd.to_datetime(raw.index)
    hist = raw.tail(PLOT_HISTORY_DAYS).copy()
    latest_date = hist.index[-1]
    latest_close = float(hist["Close"].iloc[-1])
    combined_fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist["Close"] / latest_close,
        mode="lines",
        name=f"{ticker} real normalized Close",
        line=dict(width=3)
    ))

    for model_name in MODEL_NAMES:
        fdf = forecasts_df[(forecasts_df["Ticker"] == ticker) & (forecasts_df["Model"] == model_name)].copy()
        if fdf.empty:
            continue
        joined_x = [pd.Timestamp(fdf["Latest_Date"].iloc[0])] + list(fdf["Date"])
        joined_y = [1.0] + list(fdf["Predicted_Close"] / float(fdf["Latest_Close"].iloc[0]))
        combined_fig.add_trace(go.Scatter(
            x=joined_x,
            y=joined_y,
            mode="lines",
            name=f"{ticker} {model_name} forecast",
            line=dict(width=1.7)
        ))

combined_fig.update_layout(
    title="AAPL and MSFT: Normalized Real Prices and 2-Month Forecasts from All Models",
    xaxis_title="Date",
    yaxis_title="Normalized price: latest real close = 1.0",
    legend_title="Series",
    template="plotly_white",
    width=1300,
    height=750
)
save_plotly_png(combined_fig, "AAPL_MSFT_normalized_two_month_forecast_all_models.png")

print("\nForecasting cell completed.")
print("Interpretation note: these are experimental recursive forecasts, not investment advice.")
print("A useful model should beat the naive baseline on validation Theil's U (< 1 is better than zero-return baseline).")
